# DS‑1 · режимы маршрутов 7/50 и погода

Запустите **Run All** в Kaggle: CPU, Internet включён. Исходные метки
скачиваются из архива организаторов, настоящая погода Москвы за 2025 год
уже встроена. Исходный CSV с результатом **0,877** воспроизводится отдельно.

**Что изменено:**
- Маршрут 50: отмена по выходным с 6 сентября; восстановление с 15 ноября.
- Маршрут 7: июльское укорочение, августовская конечная, новая конечная
  с 6 сентября, дополнительный объезд 20 сентября–3 октября, восстановление
  с 15 ноября. Это разные состояния, а не один коэффициент на весь период.
- Профиль соответствующего режима получает вес 25%; редкий/неизвестный
  режим сохраняет базовый прогноз. Положительные метки не перезаписываются.
- Погода сравнивается с таким же обучением без погодных признаков,
  отдельно с режимами и без них. Неудачные поправки отклоняются по маршрутам.

Режим **competition**: фактическая погода и поздние транспортные новости
разрешены организаторами. Это не операционный прогноз на дату 31 октября.
Никаких фактических пассажирских значений из будущего модель не использует.

Просмотрен весь `docs/research` ветки `ds2/research-protocol`:
19 Markdown + 19 Python + 18 JSON = **56 файлов**; проверены 1063 равенства
сохранённой метрики. Ранее названные 57 включали каталог `results`.
Это аудит исследований, а не заявление об их повторном запуске.

После выполнения скачайте **submission_network_weather.csv** и
**network_weather_report.json**. Новый результат на платформе неизвестен.
Сентябрь–октябрь уже использовались для подбора — локальный score исследовательский.


## 1. Настройки
Ссылка уже указана. Режим `yandex` скачивает архив автоматически; `input` оставлен для ранее загруженных файлов.


In [ ]:
import os
import sys
import json
import csv
import hashlib
import subprocess
import tempfile
import zipfile
import importlib.metadata
from pathlib import Path

PROFILES_ONLY = True  # Базовые профили фиксированы; погодные поправки обучаются LightGBM.
RUN_TESTS = True
DATA_SOURCE = 'yandex'  # 'input' — необязательный ручной вариант через Add Input.
YANDEX_URL = 'https://disk.yandex.ru/d/DiFwlfMOauxjBg'
DOWNLOAD_CACHE = '/tmp/mthack_yandex'
INPUT_ROOT = Path(os.environ.get('MTHACK_INPUT_DIR', '/kaggle/input'))
WORK = Path(os.environ.get('MTHACK_WORK_DIR', '/kaggle/working'))
INPUT_FILES = {
    'train': '',   # например /kaggle/input/my-private-data/labels_day_train.csv
    'test': '',    # labels_day_test.csv — фактические метки сентября–октября
    'sample': '',  # исходный test_submission.csv за ноябрь–декабрь
}
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
if sys.version_info < (3, 11):
    raise RuntimeError('Нужен Python 3.11+; выберите актуальную Python-среду Kaggle.')
WORK.mkdir(parents=True, exist_ok=True)
RUNTIME = Path(tempfile.mkdtemp(prefix='mthack_ds1_code_'))
EXTRACTED = Path(tempfile.mkdtemp(prefix='mthack_ds1_inputs_'))
OUT = WORK / 'network_weather_results'
print('Python:', sys.version.split()[0])
if not PROFILES_ONLY:
    for package in ['numpy', 'scipy', 'scikit-learn', 'lightgbm', 'catboost']:
        try:
            print(package, importlib.metadata.version(package))
        except importlib.metadata.PackageNotFoundError:
            print(package, 'не установлен (для обучения бустинга установите пакет или PROFILES_ONLY=True)')
print('Результаты:', OUT)
try:
    import lightgbm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lightgbm'])


## 2. Встроенный код модели
Ячейка разворачивает точную копию модулей DS‑1 во временной папке.
Ничего не скачивается. Исходные данные не входят в ноутбук.


In [ ]:
SOURCES = {'ml/forecast/core.py': '"""Strict readers and full-grid scoring of organizer hourly labels (MSK)."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport math\nfrom collections import defaultdict\nfrom datetime import date, timedelta\nfrom pathlib import Path\n\nKey = tuple[str, date, int]\nLabels = dict[Key, int]\n\nRUSSIAN_HOLIDAYS_2025 = frozenset({\n    date(2025, 1, 1), date(2025, 1, 2), date(2025, 1, 3), date(2025, 1, 4),\n    date(2025, 1, 5), date(2025, 1, 6), date(2025, 1, 7), date(2025, 1, 8),\n    date(2025, 2, 23), date(2025, 3, 8),\n    date(2025, 5, 1), date(2025, 5, 2), date(2025, 5, 8), date(2025, 5, 9),\n    date(2025, 6, 12), date(2025, 6, 13),\n    date(2025, 11, 3), date(2025, 11, 4),\n    date(2025, 12, 31),\n})\n\nRUSSIAN_WORKING_WEEKENDS_2025 = frozenset({\n    date(2025, 11, 1),\n})\n\n\ndef is_workday(d: date) -> bool:\n    if d in RUSSIAN_WORKING_WEEKENDS_2025:\n        return True\n    if d in RUSSIAN_HOLIDAYS_2025:\n        return False\n    return d.weekday() < 5\n\n\ndef effective_weekday(d: date) -> int:\n    if d in RUSSIAN_WORKING_WEEKENDS_2025:\n        return 4\n    if d in RUSSIAN_HOLIDAYS_2025:\n        return 6\n    return d.weekday()\n\n\ndef calendar_day_type(d: date) -> int:\n    if not is_workday(d):\n        return 3 if (effective_weekday(d) == 6 or d in RUSSIAN_HOLIDAYS_2025) else 2\n    if d in RUSSIAN_WORKING_WEEKENDS_2025:\n        return 4\n    if d.weekday() == 4:\n        return 1\n    return 0\n\n\ndef days(start: date, end: date):\n    while start <= end:\n        yield start\n        start += timedelta(days=1)\n\n\ndef grid(routes, start, end):\n    return [(str(r), d, h) for r in routes for d in days(start, end) for h in range(24)]\n\n\ndef parse_key(row) -> Key:\n    route = row[\'route\']\n    if not route or not route.isascii() or not route.isdecimal() or int(route) <= 0:\n        raise ValueError(f\'Invalid route: {route!r}\')\n    day = date.fromisoformat(row[\'date\'])\n    if day.isoformat() != row[\'date\']:\n        raise ValueError(\'Date must be YYYY-MM-DD\')\n    hour = int(row[\'hour\'])\n    if str(hour) != row[\'hour\'] or not 0 <= hour <= 23:\n        raise ValueError(f\'Invalid hour: {row["hour"]!r}\')\n    return str(int(route)), day, hour\n\n\ndef load_labels(paths) -> Labels:\n    result = {}\n    for path in paths:\n        with Path(path).open(encoding=\'utf-8-sig\', newline=\'\') as f:\n            reader = csv.DictReader(f, delimiter=\';\')\n            if reader.fieldnames != [\'route\', \'date\', \'hour\', \'boardings\']:\n                raise ValueError(f\'Unexpected label schema: {reader.fieldnames}\')\n            for row in reader:\n                if None in row or any(v is None for v in row.values()):\n                    raise ValueError(\'Malformed CSV row\')\n                key = parse_key(row)\n                value = int(row[\'boardings\'])\n                if value < 0 or str(value) != row[\'boardings\']:\n                    raise ValueError(\'Boardings must be nonnegative integers\')\n                if key in result:\n                    raise ValueError(f\'Duplicate label key across input files: {key}\')\n                result[key] = value\n    if not result:\n        raise ValueError(\'Labels are empty\')\n    return result\n\n\ndef rounded(value) -> int:\n    value = float(value)\n    if not math.isfinite(value) or abs(value) > 10**12:\n        raise ValueError(\'Prediction must be finite and within 1e12\')\n    return max(0, round(value))  # Same half-to-even rule as DS-2 baseline.\n\n\ndef sha256(path):\n    with Path(path).open(\'rb\') as f:\n        return hashlib.file_digest(f, \'sha256\').hexdigest()\n\n\ndef score(keys, truth, predictions, peak_hours=None):\n    """Score exactly the requested grid; absent label = 0 is an explicit assumption."""\n    if len(keys) != len(set(keys)) or set(keys) != set(predictions):\n        raise ValueError(\'Prediction keys must match the unique evaluation grid exactly\')\n    if set(truth) - set(keys):\n        raise ValueError(\'Truth contains keys outside evaluation grid\')\n    groups = defaultdict(lambda: {\'rows\': 0, \'actual_sum\': 0, \'absolute_error_sum\': 0,\n                                  \'pred_minus_actual\': 0, \'missing_label_rows\': 0})\n    for key in keys:\n        route, day, hour = key\n        actual, pred = truth.get(key, 0), rounded(predictions[key])\n        if actual < 0 or not math.isfinite(actual):\n            raise ValueError(\'Invalid truth\')\n        names = [\'all\', f\'route:{route}\', f\'month:{day:%Y-%m}\',\n                 \'hours:day_06_22\' if 6 <= hour < 23 else \'hours:night_23_05\',\n                 \'labels:observed\' if key in truth else \'labels:missing_assumed_zero\']\n        if peak_hours is not None:\n            names.append(\'hours:train_peaks\' if hour in peak_hours.get(route, ())\n                         else \'hours:other\')\n        for name in names:\n            m = groups[name]\n            m[\'rows\'] += 1\n            m[\'actual_sum\'] += actual\n            m[\'absolute_error_sum\'] += abs(pred - actual)\n            m[\'pred_minus_actual\'] += pred - actual\n            m[\'missing_label_rows\'] += key not in truth\n    if not groups[\'all\'][\'actual_sum\']:\n        raise ValueError(\'Global WAPE undefined: sum(y) must be positive\')\n    for m in groups.values():\n        m[\'wape\'] = m[\'absolute_error_sum\'] / m[\'actual_sum\'] if m[\'actual_sum\'] else None\n        m[\'wape_score\'] = max(0, 1 - m[\'wape\']) if m[\'wape\'] is not None else None\n    return dict(sorted(groups.items()))\n\n\ndef peak_hours(history, routes):\n    totals = defaultdict(int)\n    for (r, _, h), y in history.items():\n        totals[r, h] += y\n    return {r: sorted(range(24), key=lambda h: (-totals[r, h], h))[:4] for r in routes}\n', 'ml/forecast/models.py': '"""Models receive ONLY history through cutoff. Predict never accepts future truth."""\nfrom __future__ import annotations\n\nimport math\nfrom collections import defaultdict\nfrom datetime import date, timedelta\nfrom statistics import median, mean\n\nfrom .core import grid, rounded, is_workday, effective_weekday, calendar_day_type\n\n\nclass Profile:\n    def __init__(self, statistic=\'mean\', missing=\'zero\', shrink=0.2, window=None,\n                 use_calendar=False, route50_rule=False):\n        if statistic not in (\'mean\', \'median\') or missing not in (\'zero\', \'observed\'):\n            raise ValueError(\'Invalid profile configuration\')\n        self.statistic, self.missing, self.shrink, self.window = statistic, missing, shrink, window\n        self.use_calendar = use_calendar\n        self.route50_rule = route50_rule\n\n    def fit(self, history, start, cutoff):\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'Training history crosses cutoff or is empty\')\n        self.cutoff = cutoff\n        self.routes = sorted({k[0] for k in history})\n        if self.window:\n            start = max(start, cutoff - timedelta(days=self.window - 1))\n        filtered = {k: v for k, v in history.items() if k[1] >= start}\n        local, hour, route = defaultdict(list), defaultdict(list), defaultdict(list)\n        keys = grid(self.routes, start, cutoff) if self.missing == \'zero\' else filtered\n        for r, d, h in keys:\n            y = filtered.get((r, d, h), 0)\n            w = effective_weekday(d) if self.use_calendar else d.weekday()\n            local[r, w, h].append(y)\n            hour[r, h].append(y)\n            route[r].append(y)\n        fn = mean if self.statistic == \'mean\' else median\n        self.local = {k: fn(v) for k, v in local.items()}\n        self.hour = {k: fn(v) for k, v in hour.items()}\n        self.route = {k: fn(v) for k, v in route.items()}\n        return self\n\n    def predict(self, keys):\n        if any(k[1] <= self.cutoff for k in keys):\n            raise ValueError(\'Prediction dates must be after cutoff\')\n        result = {}\n        for key in keys:\n            r, d, h = key\n            if self.route50_rule and r == \'50\' and not is_workday(d) and date(2025, 9, 6) <= d < date(2025, 11, 15):\n                result[key] = 0\n                continue\n            w = effective_weekday(d) if self.use_calendar else d.weekday()\n            parent = self.hour.get((r, h), self.route.get(r, 0))\n            local = self.local.get((r, w, h), parent)\n            result[key] = rounded((1 - self.shrink) * local + self.shrink * parent)\n        return result\n\n\nclass CalendarProfile:\n    """Optimized calendar profile: blends recent 56-day trend with full history on effective weekdays."""\n    def __init__(self, recent_weight=0.6, window=56, route50_rule=True):\n        self.recent_weight = recent_weight\n        self.window = window\n        self.route50_rule = route50_rule\n\n    def fit(self, history, start, cutoff):\n        self.cutoff = cutoff\n        self.full_profile = Profile(\'median\', shrink=0.0, use_calendar=True,\n                                    route50_rule=self.route50_rule).fit(history, start, cutoff)\n        self.recent_profile = Profile(\'median\', shrink=0.0, window=self.window,\n                                      use_calendar=True, route50_rule=self.route50_rule).fit(history, start, cutoff)\n        return self\n\n    def predict(self, keys):\n        p_full = self.full_profile.predict(keys)\n        p_recent = self.recent_profile.predict(keys)\n        w = self.recent_weight\n        return {k: rounded(w * p_recent[k] + (1 - w) * p_full[k]) for k in keys}\n\n\nclass Boosting:\n    def __init__(self, kind=\'hgb\', recursive=False, use_calendar=True, route50_rule=False):\n        self.kind = kind\n        self.recursive = recursive\n        self.use_calendar = use_calendar\n        self.route50_rule = route50_rule\n\n    def _extract_profiles(self, history, start, cutoff):\n        full_med, full_mean = defaultdict(list), defaultdict(list)\n        rec_med, rec_mean = defaultdict(list), defaultdict(list)\n        hour_med = defaultdict(list)\n        rec_start = max(start, cutoff - timedelta(days=55))\n\n        keys = grid(self.routes, start, cutoff)\n        for r, d, h in keys:\n            y = history.get((r, d, h), 0)\n            eff_w = effective_weekday(d) if self.use_calendar else d.weekday()\n            full_med[r, eff_w, h].append(y)\n            full_mean[r, eff_w, h].append(y)\n            hour_med[r, h].append(y)\n            if d >= rec_start:\n                rec_med[r, eff_w, h].append(y)\n                rec_mean[r, eff_w, h].append(y)\n\n        self.p_full_med = {k: median(v) for k, v in full_med.items()}\n        self.p_full_mean = {k: mean(v) for k, v in full_mean.items()}\n        self.p_rec_med = {k: median(v) for k, v in rec_med.items()}\n        self.p_rec_mean = {k: mean(v) for k, v in rec_mean.items()}\n        self.p_hour_med = {k: median(v) for k, v in hour_med.items()}\n\n    def features(self, keys, context=None):\n        import numpy as np\n        result = []\n        if self.recursive:\n            for r, d, h in keys:\n                t = d.timetuple().tm_yday\n                row = [self.route_codes[r], d.weekday(), h, d.month, t,\n                       int(d.weekday() >= 5), math.sin(2 * math.pi * t / 365),\n                       math.cos(2 * math.pi * t / 365)]\n                if context is not None:\n                    for lag in (1, 7):\n                        key = r, d - timedelta(days=lag), h\n                        row.extend([context.get(key, 0), int(key in context)])\n                result.append(row)\n            return np.asarray(result, dtype=float)\n\n        for r, d, h in keys:\n            t = d.timetuple().tm_yday\n            eff_w = effective_weekday(d) if self.use_calendar else d.weekday()\n            workday = int(is_workday(d)) if self.use_calendar else int(d.weekday() < 5)\n            dtype = calendar_day_type(d) if self.use_calendar else (2 if d.weekday() >= 5 else 0)\n\n            pf_med = getattr(self, \'p_full_med\', {}).get((r, eff_w, h), getattr(self, \'p_hour_med\', {}).get((r, h), 0))\n            pf_mean = getattr(self, \'p_full_mean\', {}).get((r, eff_w, h), pf_med)\n            pr_med = getattr(self, \'p_rec_med\', {}).get((r, eff_w, h), pf_med)\n            pr_mean = getattr(self, \'p_rec_mean\', {}).get((r, eff_w, h), pf_mean)\n            ph_med = getattr(self, \'p_hour_med\', {}).get((r, h), 0)\n            trend_ratio = pr_mean / (pf_mean + 1.0) if pf_mean > 0 else 1.0\n\n            is_r50_susp = int(self.route50_rule and r == \'50\' and not is_workday(d)\n                              and date(2025, 9, 6) <= d < date(2025, 11, 15))\n\n            row = [\n                h,\n                d.weekday(),\n                eff_w,\n                dtype,\n                workday,\n                d.month,\n                d.day,\n                t,\n                math.sin(2 * math.pi * h / 24),\n                math.cos(2 * math.pi * h / 24),\n                math.sin(2 * math.pi * t / 365),\n                math.cos(2 * math.pi * t / 365),\n                pf_med,\n                pf_mean,\n                pr_med,\n                pr_mean,\n                ph_med,\n                trend_ratio,\n                is_r50_susp,\n            ]\n            result.append(row)\n        return np.asarray(result, dtype=float)\n\n    def fit(self, history, start, cutoff):\n        import numpy as np\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'Training history crosses cutoff or is empty\')\n        self.cutoff = cutoff\n        self.routes = sorted({k[0] for k in history})\n        self.route_codes = {r: i for i, r in enumerate(self.routes)}\n        self._extract_profiles(history, start, cutoff)\n        self.history = dict(history)\n\n        keys = grid(self.routes, start, cutoff)\n        if self.recursive:\n            from sklearn.ensemble import HistGradientBoostingRegressor\n            keys = [k for k in keys if k[1] >= start + timedelta(days=7)]\n            self.model = HistGradientBoostingRegressor(\n                loss=\'absolute_error\', learning_rate=.08, max_iter=150,\n                max_leaf_nodes=15, min_samples_leaf=30, l2_regularization=2,\n                categorical_features=[0, 1, 2, 3, 5], early_stopping=False, random_state=2025)\n            self.model.fit(self.features(keys, history), [history.get(k, 0) for k in keys])\n            return self\n\n        # Out-of-time target aggregates: each monthly block uses ONLY prior days.\n        # Reuse the exact feature builder used at inference, with an earlier cutoff.\n        train_keys, train_features = [], []\n        block_start = start + timedelta(days=28)\n        while block_start <= cutoff:\n            next_month = (block_start.replace(day=28) + timedelta(days=4)).replace(day=1)\n            block_end = min(cutoff, next_month - timedelta(days=1))\n            prefix_end = block_start - timedelta(days=1)\n            prefix = {k:v for k,v in history.items() if k[1] <= prefix_end}\n            self._extract_profiles(prefix, start, prefix_end)\n            batch = grid(self.routes, block_start, block_end)\n            train_keys.extend(batch)\n            train_features.extend(self.features(batch))\n            block_start = block_end + timedelta(days=1)\n        if not train_keys:\n            raise ValueError(\'Direct boosting needs at least 29 days of history\')\n        # Final prediction profiles may now use all history up to the actual cutoff.\n        self._extract_profiles(history, start, cutoff)\n        route_keys, route_features = defaultdict(list), defaultdict(list)\n        for k, features in zip(train_keys, train_features):\n            route_keys[k[0]].append(k)\n            route_features[k[0]].append(features)\n\n        self.models = {}\n        for r in self.routes:\n            r_keys = route_keys[r]\n            X = np.asarray(route_features[r], dtype=float)\n            y = np.array([history.get(k, 0) for k in r_keys], dtype=float)\n\n            if self.kind == \'lgb\':\n                try:\n                    import lightgbm as lgb\n                    model = lgb.LGBMRegressor(\n                        objective=\'regression_l1\', metric=\'mae\', learning_rate=0.03,\n                        n_estimators=300, num_leaves=15, min_child_samples=10,\n                        subsample=0.85, colsample_bytree=0.85, random_state=2025,\n                        verbose=-1, n_jobs=2\n                    )\n                    model.fit(X, y)\n                    self.models[r] = model\n                    continue\n                except Exception as exc:\n                    raise RuntimeError(f\'{self.kind} training failed for route {r}; no silent fallback\') from exc\n\n            if self.kind == \'catboost\':\n                try:\n                    import catboost as cb\n                    model = cb.CatBoostRegressor(\n                        loss_function=\'MAE\', eval_metric=\'MAE\', iterations=300,\n                        learning_rate=0.04, depth=5, random_seed=2025, verbose=0,\n                        thread_count=2\n                    )\n                    model.fit(X, y)\n                    self.models[r] = model\n                    continue\n                except Exception as exc:\n                    raise RuntimeError(f\'{self.kind} training failed for route {r}; no silent fallback\') from exc\n\n            if self.kind == \'blend\':\n                submodels = []\n                try:\n                    import lightgbm as lgb\n                    m1 = lgb.LGBMRegressor(\n                        objective=\'regression_l1\', metric=\'mae\', learning_rate=0.03,\n                        n_estimators=300, num_leaves=15, min_child_samples=10,\n                        subsample=0.85, colsample_bytree=0.85, random_state=2025,\n                        verbose=-1, n_jobs=2\n                    )\n                    m1.fit(X, y)\n                    submodels.append((\'lgb\', m1, 0.5))\n                except Exception as exc:\n                    raise RuntimeError(f\'{self.kind} training failed for route {r}; no silent fallback\') from exc\n\n                try:\n                    import catboost as cb\n                    m2 = cb.CatBoostRegressor(\n                        loss_function=\'MAE\', eval_metric=\'MAE\', iterations=300,\n                        learning_rate=0.04, depth=5, random_seed=2025, verbose=0,\n                        thread_count=2\n                    )\n                    m2.fit(X, y)\n                    submodels.append((\'catboost\', m2, 0.5))\n                except Exception as exc:\n                    raise RuntimeError(f\'{self.kind} training failed for route {r}; no silent fallback\') from exc\n\n                if not submodels:\n                    from sklearn.ensemble import HistGradientBoostingRegressor\n                    m3 = HistGradientBoostingRegressor(\n                        loss=\'absolute_error\', learning_rate=0.04, max_iter=200,\n                        max_leaf_nodes=15, min_samples_leaf=10, l2_regularization=1,\n                        early_stopping=False, random_state=2025\n                    )\n                    m3.fit(X, y)\n                    submodels.append((\'hgb\', m3, 1.0))\n                self.models[r] = submodels\n                continue\n\n            from sklearn.ensemble import HistGradientBoostingRegressor\n            model = HistGradientBoostingRegressor(\n                loss=\'absolute_error\', learning_rate=0.04, max_iter=200,\n                max_leaf_nodes=15, min_samples_leaf=10, l2_regularization=1,\n                early_stopping=False, random_state=2025\n            )\n            model.fit(X, y)\n            self.models[r] = model\n\n        return self\n\n    def _predict_route(self, r, X):\n        m = getattr(self, \'models\', {}).get(r)\n        if m is None:\n            if hasattr(self, \'model\'):\n                return self.model.predict(X)\n            return [0] * len(X)\n        if isinstance(m, list):\n            total_w = sum(w for _, _, w in m)\n            preds = sum(w * sub.predict(X) for _, sub, w in m) / total_w\n            return preds\n        return m.predict(X)\n\n    def predict(self, keys):\n        if any(k[1] <= self.cutoff for k in keys):\n            raise ValueError(\'Prediction dates must be after cutoff\')\n        known = [k for k in keys if k[0] in self.route_codes]\n        result = {k: 0 for k in keys}\n        if not known:\n            return result\n\n        if not self.recursive:\n            route_known = defaultdict(list)\n            for k in known:\n                route_known[k[0]].append(k)\n            for r, r_keys in route_known.items():\n                X = self.features(r_keys)\n                raw_preds = self._predict_route(r, X)\n                for k, p in zip(r_keys, raw_preds):\n                    if self.route50_rule and r == \'50\' and not is_workday(k[1]) and date(2025, 9, 6) <= k[1] < date(2025, 11, 15):\n                        result[k] = 0\n                    else:\n                        result[k] = rounded(p)\n            return result\n\n        context = dict(self.history)\n        day, end = self.cutoff + timedelta(days=1), max(k[1] for k in known)\n        wanted = set(known)\n        while day <= end:\n            batch = grid(self.routes, day, day)\n            predictions = self.model.predict(self.features(batch, context))\n            for key, value in zip(batch, predictions):\n                context[key] = rounded(value)\n                if key in wanted:\n                    result[key] = context[key]\n            day += timedelta(days=1)\n        return result\n\n\ndef candidates(include_boosting=True):\n    result = {\n        \'mean_zero\': lambda: Profile(\'mean\'),\n        \'median_zero\': lambda: Profile(\'median\'),\n        \'mean_observed\': lambda: Profile(\'mean\', \'observed\'),\n        \'median_observed\': lambda: Profile(\'median\', \'observed\'),\n        \'mean_56d\': lambda: Profile(\'mean\', window=56),\n        \'profile_calendar\': lambda: CalendarProfile(recent_weight=0.6, window=56, route50_rule=True),\n    }\n    if include_boosting:\n        result.update(\n            hgb_direct=lambda: Boosting(kind=\'hgb\'),\n            lgb_direct=lambda: Boosting(kind=\'lgb\'),\n            catboost_direct=lambda: Boosting(kind=\'catboost\'),\n            blend_ensemble=lambda: Boosting(kind=\'blend\'),\n            hgb_recursive=lambda: Boosting(recursive=True),\n        )\n    return result\n\n', 'ml/forecast/route_specific.py': '"""DS-2-informed route-specific selection for a fixed 61-day forecast.\n\nNo validation labels enter fitted candidates. Selection scores are exploratory:\nSep-Oct has already been examined repeatedly by DS-2.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport importlib.util\nimport json\nimport pickle\nimport time\nfrom collections import defaultdict\nfrom datetime import date, timedelta\nfrom pathlib import Path\nimport numpy as np\nfrom .core import days, grid, load_labels, rounded, score, sha256, effective_weekday\nfrom .models import Profile, Boosting\nfrom .run import FIRST, FINAL_CUTOFF, FOLDS, dump\nfrom ml.submit.adapter import read_sample, write_submission, export_api, validate\n\nRESEARCH_COMMIT = \'b593ebb\'\nEXPECTED_HASHES = {\n    \'train\': \'20a61009b4ee00327784c6fa888109609fd3a1905ea3a8509b36964dd9baced7\',\n    \'test\': \'37479acea3e319ac613d72df7217bbe57b66af49644e538139174fd534ce2347\',\n    \'sample\': \'71c8126bb51068c936462a170da885bad860fc91a58b3d2ac731876d324d7ec3\',\n}\n\n\ndef verify_inputs(paths, synthetic=False):\n    result = {role: {\'name\': p.name, \'sha256\': sha256(p)} for role, p in paths.items()}\n    if not synthetic:\n        for role, item in result.items():\n            if item[\'sha256\'] != EXPECTED_HASHES[role]:\n                raise ValueError(f\'{role}: SHA256 does not match real DS-2 inputs. \'\n                                 \'Do not train a submission on synthetic or unverified files. \'\n                                 f"Received {item[\'sha256\']}; expected {EXPECTED_HASHES[role]}")\n    return result\n\n\nclass SeasonalProfile:\n    """Calendar regimes, robust hourly shapes, optional local history blend.\n\nSummer weekdays are a separate regime, not deleted unconditionally: if a\nforecast requests summer, use summer history when at least 3 dates exist.\nFor autumn this reproduces the DS-2 school-term exclusion candidate.\n"""\n    def __init__(self, statistic=\'median\', shrink=.2, window=None,\n                 seasonal=True, calendar=True, recent_weight=0., regime7=False):\n        self.statistic, self.shrink, self.window = statistic, shrink, window\n        self.seasonal, self.calendar = seasonal, calendar\n        self.recent_weight, self.regime7 = recent_weight, regime7\n\n    @staticmethod\n    def summer(d):\n        return d.month in (7, 8) and d.weekday() < 5\n\n    def fit(self, history, start, cutoff):\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'History empty or crosses cutoff\')\n        self.cutoff = cutoff\n        self.routes = sorted({k[0] for k in history}, key=int)\n        self.tables = {}\n        # All keys are explicit, including absent training labels = zero assumption.\n        for r in self.routes:\n            for season in (None, False, True):\n                ds = [d for d in days(start, cutoff)\n                      if (not self.window or (cutoff-d).days < self.window)\n                      and (season is None or self.summer(d) == season)]\n                local, parent = defaultdict(list), defaultdict(list)\n                recent = defaultdict(list)\n                regime = defaultdict(list)\n                for d in ds:\n                    w = effective_weekday(d) if self.calendar else d.weekday()\n                    for h in range(24):\n                        y = history.get((r, d, h), 0)\n                        local[w,h].append(y); parent[h].append(y)\n                        if (cutoff-d).days < 56:\n                            recent[w,h].append(y)\n                        if r == \'7\' and cutoff >= date(2025,8,16) and d >= date(2025,8,16):\n                            regime[w,h].append(y)\n                fn = np.mean if self.statistic == \'mean\' else np.median\n                self.tables[r,season] = tuple({k: float(fn(v)) for k,v in t.items() if len(v) >= 3}\n                                             for t in (local,parent,recent,regime))\n        return self\n\n    def predict(self, keys):\n        if any(d <= self.cutoff for _,d,_ in keys):\n            raise ValueError(\'Prediction dates must follow cutoff\')\n        out = {}\n        for key in keys:\n            r,d,h = key\n            if r not in self.routes:\n                out[key] = 0; continue\n            season = self.summer(d) if self.seasonal else None\n            loc,par,rec,reg = self.tables[r,season]\n            fallback,fp,_,_ = self.tables[r,None]\n            w = effective_weekday(d) if self.calendar else d.weekday()\n            parent = par.get(h,fp.get(h,0))\n            local = loc.get((w,h),fallback.get((w,h),parent))\n            p = (1-self.shrink)*local + self.shrink*parent\n            if (w,h) in rec:\n                p = (1-self.recent_weight)*p + self.recent_weight*rec[w,h]\n            if self.regime7 and r == \'7\' and d.weekday() >= 5 and (w,h) in reg:\n                # DS-2 experiment 13: 25% sparse post-Aug16 profile, no future notice.\n                p = .75*p + .25*reg[w,h]\n            out[key] = rounded(p)\n        return out\n\n\ndef candidate_factories(profiles_only=False):\n    pool = {\n        \'median_zero\': lambda: Profile(\'median\'),\n        \'mean_zero\': lambda: Profile(\'mean\'),\n        \'school_ds2\': lambda: SeasonalProfile(calendar=False),\n        \'school_calendar\': lambda: SeasonalProfile(),\n        \'school_local\': lambda: SeasonalProfile(shrink=0),\n        \'school_mean\': lambda: SeasonalProfile(statistic=\'mean\'),\n        \'school_recent25\': lambda: SeasonalProfile(recent_weight=.25),\n        \'school_recent50\': lambda: SeasonalProfile(recent_weight=.5),\n        \'school_route7\': lambda: SeasonalProfile(regime7=True),\n        \'median_56d\': lambda: SeasonalProfile(window=56, seasonal=False),\n        \'median_112d\': lambda: SeasonalProfile(window=112, seasonal=False),\n        \'mean_56d\': lambda: SeasonalProfile(statistic=\'mean\',window=56,seasonal=False),\n        \'mean_112d\': lambda: SeasonalProfile(statistic=\'mean\',window=112,seasonal=False),\n    }\n    if not profiles_only:\n        pool.update({\n            \'hgb_causal\': lambda: Boosting(kind=\'hgb\',route50_rule=False),\n            \'lgb_causal\': lambda: Boosting(kind=\'lgb\',route50_rule=False),\n            \'catboost_causal\': lambda: Boosting(kind=\'catboost\',route50_rule=False),\n        })\n    return pool\n\n\ndef choose_routes(keys, truth, predictions, routes, min_gain=.01):\n    """Conservative route choice: beat school baseline in both calendar months.\n\nA predeclared 50/50 blend is also considered. No continuous weight fitting.\nFall back to school baseline unless each month improves by >= 1% in MAE.\n"""\n    months = sorted({d.month for _,d,_ in keys})\n    choices, diagnostics = {}, {}\n    for r in routes:\n        rk = [k for k in keys if k[0] == r]\n        by_month = [[k for k in rk if k[1].month == m] for m in months]\n        def losses(name, weight):\n            return [sum(abs(rounded(weight*predictions[name][k] +\n                                (1-weight)*predictions[\'school_calendar\'][k])-truth.get(k,0))\n                        for k in batch) for batch in by_month]\n        base = losses(\'school_calendar\',1.)\n        options = [(\'school_calendar\',1.,base)]\n        diagnostics[r] = {\'baseline_month_errors\':base,\'candidates\':{}}\n        for name in predictions:\n            for weight in (0.5,1.):\n                errs = losses(name,weight)\n                diagnostics[r][\'candidates\'][f\'{name}:{weight}\'] = errs\n                if all(e <= b*(1-min_gain) for e,b in zip(errs,base)) and sum(errs)<sum(base):\n                    options.append((name,weight,errs))\n        name,weight,errs = min(options,key=lambda item:(sum(item[2]),item[0],item[1]))\n        choices[r] = {\'candidate\':name,\'weight\':weight,\'month_errors\':errs}\n    return choices,diagnostics\n\n\ndef combine(keys, predictions, choices):\n    result = {}\n    for key in keys:\n        choice = choices.get(key[0])\n        if choice is None:\n            result[key] = 0\n        else:\n            name,w = choice[\'candidate\'],choice[\'weight\']\n            result[key] = rounded(w*predictions[name][key]+(1-w)*predictions[\'school_calendar\'][key])\n    return result\n\n\nclass RouteEnsemble:\n    def __init__(self, models, choices, cutoff):\n        self.models,self.choices,self.cutoff = models,choices,cutoff\n    def predict(self,keys):\n        return combine(keys,{name:m.predict(keys) for name,m in self.models.items()},self.choices)\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__)\n    for name in (\'train\',\'test\',\'sample\',\'out\'):p.add_argument(\'--\'+name,type=Path,required=True)\n    p.add_argument(\'--profiles-only\',action=\'store_true\')\n    p.add_argument(\'--synthetic\',action=\'store_true\')\n    args=p.parse_args(argv)\n    inputs=verify_inputs({n:getattr(args,n) for n in (\'train\',\'test\',\'sample\')},args.synthetic)\n    train,test=load_labels([args.train]),load_labels([args.test])\n    if any(not FIRST<=k[1]<=date(2025,8,31) for k in train):raise ValueError(\'Invalid train period\')\n    if any(not date(2025,9,1)<=k[1]<=FINAL_CUTOFF for k in test):raise ValueError(\'Invalid test period\')\n    values=load_labels([args.train,args.test]); _,sample=read_sample(args.sample)\n    routes=sorted({k[0] for k in values},key=int)\n    if set(routes)-{k[0] for k in sample}:raise ValueError(\'Sample misses known routes\')\n    for lo,hi,part in [(FIRST,date(2025,8,31),train),(date(2025,9,1),FINAL_CUTOFF,test)]:\n        if {k[1] for k in part} != set(days(lo,hi)):raise ValueError(\'Missing complete input dates\')\n    if not args.profiles_only:\n        missing=[n for n in (\'sklearn\',\'lightgbm\',\'catboost\') if importlib.util.find_spec(n) is None]\n        if missing:raise RuntimeError(\'Missing training packages: \'+\', \'.join(missing)+\'. Install them or use --profiles-only explicitly.\')\n    factories=candidate_factories(args.profiles_only)\n    args.out.mkdir(parents=True,exist_ok=True)\n    report={\'schema_version\':2,\'research_commit\':RESEARCH_COMMIT,\'inputs\':inputs,\n            \'data_kind\':\'synthetic_not_for_submission\' if args.synthetic else \'verified_organizer_labels\',\n            \'selection_warning\':\'Research validation repeatedly examined in DS-2; not an independent test or platform score.\',\n            \'missing_label_policy\':\'zero-filled full grid for training and scoring; not proof of zero physical demand\',\n            \'external_policy\':\'calendar only; no future network notices, weather or validation labels in features\',\n            \'folds\':{},\'platform_score\':None,\'library_versions\':{}}\n    for pkg in (\'numpy\',\'scikit-learn\',\'lightgbm\',\'catboost\'):\n        try:report[\'library_versions\'][pkg]=importlib.metadata.version(pkg)\n        except importlib.metadata.PackageNotFoundError:pass\n    for label,cutoff,start,end in FOLDS:\n        history={k:v for k,v in values.items() if k[1]<=cutoff}\n        keys=grid(routes,start,end); truth={k:v for k,v in values.items() if start<=k[1]<=end}\n        forecasts={};report[\'folds\'][label]={}\n        for name,factory in factories.items():\n            t=time.perf_counter();model=factory().fit(history,FIRST,cutoff)\n            forecasts[name]=model.predict(keys)\n            m=score(keys,truth,forecasts[name])\n            report[\'folds\'][label][name]={\'metrics\':m,\'seconds\':round(time.perf_counter()-t,3)}\n            print(f\'{label} {name}: score={m["all"]["wape_score"]:.6f}\',flush=True)\n        if label==\'sep_oct\':\n            # Forward diagnostic: choose using September only, evaluate October\n            # from the SAME Aug31 forecasts (no October/September refitting).\n            sep=[k for k in keys if k[1].month==9];october=[k for k in keys if k[1].month==10]\n            sept_choices,_=choose_routes(sep,truth,forecasts,routes)\n            op=combine(october,forecasts,sept_choices)\n            report[\'october_forward_diagnostic\']={\'selection\':\'September only; forecasts fixed at Aug31\',\n                \'choices\':sept_choices,\'metrics\':score(october,{k:truth[k] for k in october if k in truth},op)}\n            choices,diagnostics=choose_routes(keys,truth,forecasts,routes)\n            ensemble=combine(keys,forecasts,choices)\n            report[\'folds\'][label][\'route_ensemble\']={\'metrics\':score(keys,truth,ensemble)}\n            report[\'selection\']={\'selected\':\'route_ensemble\',\'by_route\':choices,\'diagnostics\':diagnostics,\n                \'rule\':\'Per route: lower pooled Sep-Oct error AND >=1% reduction in each month vs school_calendar; weights 0.5 or 1.0.\'}\n        dump(args.out/\'report.json\',report)\n    selected_names={\'school_calendar\'}|{c[\'candidate\'] for c in choices.values()}\n    fitted={n:factories[n]().fit(values,FIRST,FINAL_CUTOFF) for n in sorted(selected_names)}\n    model=RouteEnsemble(fitted,choices,FINAL_CUTOFF);predictions=model.predict(sample)\n    report[\'code_sha256\']={q.name:sha256(q) for q in sorted(Path(__file__).parent.glob(\'*.py\'))}\n    identity=json.dumps({\'inputs\':inputs,\'choices\':choices,\'code\':report[\'code_sha256\'],\n                         \'libraries\':report[\'library_versions\']},sort_keys=True).encode()\n    version=\'ds1-route-\'+hashlib.sha256(identity).hexdigest()[:12]\n    if args.synthetic:version=\'synthetic-NOT-FOR-SUBMISSION-\'+version\n    report[\'model_version\']=version\n    report[\'cold_start\']={\'routes\':sorted({k[0] for k in sample}-set(routes),key=int),\n                          \'policy\':\'zero placeholder; unvalidated, not a required true target\'}\n    report[\'submission\']=write_submission(args.sample,predictions,args.out/\'submission.csv\')\n    export_api(sample,predictions,args.out/\'forecast_api.csv\',version)\n    (args.out/\'model.local.pkl\').write_bytes(pickle.dumps(model,protocol=5))\n    validate(args.sample,args.out/\'submission.csv\')\n    report[\'outputs_sha256\']={n:sha256(args.out/n) for n in (\'submission.csv\',\'forecast_api.csv\',\'model.local.pkl\')}\n    report[\'matches_previous_submission\'] = report[\'outputs_sha256\'][\'submission.csv\'] == \'371d05c44e20b63db110f5a5e27b8da8fafb785fdbc66374307b0ab7304a75ac\'\n    dump(args.out/\'report.json\',report)\n    print(\'Selected route models:\',json.dumps(choices,ensure_ascii=False),flush=True)\n    print(\'Validated:\',version, len(sample),\'rows; platform score unknown.\',flush=True)\n    return report\n\nif __name__==\'__main__\':\n    from ml.forecast.route_specific import main as run_main\n    run_main()\n', 'ml/forecast/test_route_specific.py': "import tempfile\nimport importlib.util\nimport unittest\nfrom datetime import date,timedelta\nfrom pathlib import Path\nfrom unittest.mock import patch\nimport numpy as np\nfrom ml.forecast.core import grid\nfrom ml.forecast.models import Boosting\nfrom ml.forecast.route_specific import SeasonalProfile,choose_routes,combine,verify_inputs,candidate_factories\n\n\nclass RouteTests(unittest.TestCase):\n    def test_unverified_inputs_rejected_before_training(self):\n        with tempfile.TemporaryDirectory() as td:\n            p=Path(td)/'fake.csv';p.write_text('synthetic')\n            with self.assertRaisesRegex(ValueError,'does not match real DS-2'):\n                verify_inputs({'train':p})\n            self.assertIn('train',verify_inputs({'train':p},synthetic=True))\n\n    def test_autumn_does_not_inherit_summer_weekday_slump(self):\n        start,cutoff=date(2025,1,1),date(2025,8,31)\n        hist={k: (20 if k[1].month in (7,8) and k[1].weekday()<5 else 100)\n              for k in grid(['1'],start,cutoff)}\n        model=SeasonalProfile(calendar=False).fit(hist,start,cutoff)\n        k=('1',date(2025,9,1),12)\n        self.assertEqual(model.predict([k])[k],100)\n        self.assertEqual(model.predict([('5',date(2025,9,1),12)])[('5',date(2025,9,1),12)],0)\n        hist['1',date(2025,9,1),12]=999\n        with self.assertRaises(ValueError):model.fit(hist,start,cutoff)\n\n    def test_route_choices_differ_and_single_month_overfit_rejected(self):\n        keys=[(r,date(2025,m,1),12) for r in ('1','7') for m in (9,10)]\n        truth={k:100 for k in keys}\n        preds={'school_calendar':dict.fromkeys(keys,80),\n               'a':{k:100 if k[0]=='1' else 70 for k in keys},\n               'b':{k:100 if k[0]=='7' else 70 for k in keys},\n               'overfit':{k:100 if k[1].month==9 else 10 for k in keys}}\n        choices,_=choose_routes(keys,truth,preds,['1','7'])\n        self.assertEqual(choices['1']['candidate'],'a')\n        self.assertEqual(choices['7']['candidate'],'b')\n        self.assertEqual(combine(keys,preds,choices),truth)\n\n    def test_boosting_target_aggregates_are_strictly_past(self):\n        start,cutoff=date(2025,1,1),date(2025,3,31)\n        hist={k:(k[1]-start).days*100+k[2] for k in grid(['1'],start,cutoff)}\n        observed=[]\n        class Spy(Boosting):\n            def _extract_profiles(self,history,start,cutoff):\n                self.profile_end=cutoff\n                return super()._extract_profiles(history,start,cutoff)\n            def features(self,keys,context=None):\n                observed.append((self.profile_end,min(k[1] for k in keys)))\n                assert self.profile_end<min(k[1] for k in keys)\n                return super().features(keys,context)\n        Spy(kind='hgb').fit(hist,start,cutoff)\n        self.assertGreaterEqual(len(observed),3)\n\n    @unittest.skipUnless(importlib.util.find_spec('lightgbm'), 'optional LightGBM')\n    def test_training_error_cannot_silently_change_library(self):\n        hist={k:100 for k in grid(['1'],date(2025,1,1),date(2025,2,28))}\n        with patch('lightgbm.LGBMRegressor.fit',side_effect=ValueError('broken fit')):\n            with self.assertRaisesRegex(RuntimeError,'no silent fallback'):\n                Boosting(kind='lgb').fit(hist,date(2025,1,1),date(2025,2,28))\n\n    def test_candidates_have_no_future_route50_rule(self):\n        for name,factory in candidate_factories().items():\n            model=factory()\n            self.assertFalse(getattr(model,'route50_rule',False),name)\n\nif __name__=='__main__':unittest.main()\n", 'ml/forecast/run.py': '"""One-command chronological backtest, model selection and exact submission."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport pickle\nimport platform\nimport time\nfrom datetime import date\nfrom pathlib import Path\n\n# A bounded CPU budget also avoids OpenMP oversubscription on desktops.\nos.environ.setdefault(\'OMP_NUM_THREADS\', \'2\')\nos.environ.setdefault(\'OPENBLAS_NUM_THREADS\', \'2\')\n\nfrom .core import grid, load_labels, peak_hours, score, sha256\nfrom .models import candidates\nfrom ml.submit.adapter import export_api, read_sample, validate, write_submission\n\nFIRST, FINAL_CUTOFF = date(2025, 1, 1), date(2025, 10, 31)\nFOLDS = [\n    (\'may_jun\', date(2025, 4, 30), date(2025, 5, 1), date(2025, 6, 30)),\n    (\'jul_aug\', date(2025, 6, 30), date(2025, 7, 1), date(2025, 8, 31)),\n    (\'sep_oct\', date(2025, 8, 31), date(2025, 9, 1), date(2025, 10, 31)),\n]\n\n\ndef dump(path, data):\n    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False) + \'\\n\',\n                          encoding=\'utf-8\')\n\n\ndef evaluate_fold(values, cutoff, start, end, factory):\n    history = {k: v for k, v in values.items() if FIRST <= k[1] <= cutoff}\n    routes = sorted({k[0] for k in history}, key=int)\n    truth = {k: v for k, v in values.items() if start <= k[1] <= end}\n    if {k[0] for k in truth} - set(routes):\n        raise ValueError(\'Validation has new routes: define cold-start evaluation explicitly\')\n    keys = grid(routes, start, end)\n    t = time.perf_counter()\n    model = factory().fit(history, FIRST, cutoff)\n    predictions = model.predict(keys)\n    metrics = score(keys, truth, predictions, peak_hours(history, routes))\n    return {\'cutoff\': str(cutoff), \'from\': str(start), \'to_inclusive\': str(end),\n            \'routes\': routes, \'metrics\': metrics,\n            \'fit_predict_score_seconds\': round(time.perf_counter() - t, 3)}\n\n\ndef select_dev(folds, names):\n    """Select on May–August only; pool absolute errors, never average route scores."""\n    losses = {name: sum(folds[fold][name][\'metrics\'][\'all\'][\'absolute_error_sum\']\n                       for fold in (\'may_jun\', \'jul_aug\')) for name in names}\n    return min(losses, key=lambda n: (losses[n], n)), losses\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--train\', required=True, type=Path)\n    parser.add_argument(\'--test\', required=True, type=Path)\n    parser.add_argument(\'--sample\', required=True, type=Path)\n    parser.add_argument(\'--out\', required=True, type=Path)\n    parser.add_argument(\'--profiles-only\', action=\'store_true\',\n                        help=\'Skip optional scikit-learn candidates explicitly\')\n    parser.add_argument(\'--synthetic\', action=\'store_true\',\n                        help=\'Mark artificial smoke data and outputs as NOT FOR SUBMISSION\')\n    args = parser.parse_args(argv)\n    from .route_specific import verify_inputs\n    verify_inputs({n:getattr(args,n) for n in (\'train\',\'test\',\'sample\')}, args.synthetic)\n    # Fail on malformed input before training or creating any outputs.\n    _, sample_keys = read_sample(args.sample)\n    train, test = load_labels([args.train]), load_labels([args.test])\n    if any(not FIRST <= k[1] <= date(2025, 8, 31) for k in train):\n        raise ValueError(\'Train must be January–August 2025 only\')\n    if any(not date(2025, 9, 1) <= k[1] <= FINAL_CUTOFF for k in test):\n        raise ValueError(\'Test labels must be September–October 2025 only\')\n    values = load_labels([args.train, args.test])\n    # Missing hours are allowed, an entirely missing date is a data availability problem.\n    for a, b, data in ((FIRST, date(2025, 8, 31), train),\n                       (date(2025, 9, 1), FINAL_CUTOFF, test)):\n        if {k[1] for k in data} != {k[1] for k in grid([\'check\'], a, b)}:\n            raise ValueError(\'Labels have missing full dates; audit before running model selection\')\n    if {k[0] for k in values} - {k[0] for k in sample_keys}:\n        raise ValueError(\'Historical routes absent from sample\')\n    models = candidates(not args.profiles_only)\n    if not args.profiles_only:\n        import sklearn  # noqa: F401; fail early instead of silently skipping boosting.\n    args.out.mkdir(parents=True, exist_ok=True)\n    inputs = {name: {\'name\': path.name, \'sha256\': sha256(path)}\n              for name, path in [(\'train\', args.train), (\'test\', args.test), (\'sample\', args.sample)]}\n    code_files = sorted(Path(__file__).parent.glob(\'*.py\')) + [\n        Path(__file__).parents[1] / \'submit\' / \'adapter.py\']\n    code = {str(p.relative_to(Path(__file__).parents[2])): sha256(p) for p in code_files}\n    installed_libs = {}\n    if not args.profiles_only:\n        for pkg in (\'numpy\', \'scipy\', \'scikit-learn\', \'lightgbm\', \'catboost\'):\n            try:\n                installed_libs[pkg] = importlib.metadata.version(pkg)\n            except importlib.metadata.PackageNotFoundError:\n                pass\n    report = {\n        \'schema_version\': 1, \'inputs\': inputs, \'code_sha256\': code,\n        \'data_kind\': \'synthetic_not_for_submission\' if args.synthetic else \'organizer_labels\',\n        \'python_version\': platform.python_version(),\n        \'library_versions\': installed_libs,\n        \'source_mode\': \'as_of_2025_10_31\', \'timezone\': \'Europe/Moscow\',\n        \'missing_label_policy\': \'zero for scoring; compare zero-filled and observed-only profiles\',\n        \'rounding\': \'clip negatives to zero; Python round half-to-even\',\n        \'peak_definition\': \'four largest hour sums per route in each training window only\',\n        \'night_definition\': \'23:00–05:59 MSK diagnostic; not an operating-hours assertion\',\n        \'external_factors\': \'Russian 2025 production calendar, route 50 weekend resumption policy\',\n        \'parameters\': {\'profile_shrink\': .2, \'short_window_days\': 56, \'recent_weight\': 0.6,\n                       \'lgb\': {\'objective\': \'regression_l1\', \'learning_rate\': .03, \'n_estimators\': 300, \'num_leaves\': 15},\n                       \'catboost\': {\'loss_function\': \'MAE\', \'iterations\': 300, \'learning_rate\': .04, \'depth\': 5},\n                       \'calendar\': \'Russian 2025 production calendar with working Saturdays and state holidays\',\n                       \'route50_policy\': \'weekend repair 2025-09-06..2025-11-14; resumed 2025-11-15\'},\n        \'folds\': {}, \'selection\': {}, \'platform_score\': None,\n    }\n    for fold, cutoff, start, end in FOLDS[:2]:\n        report[\'folds\'][fold] = {}\n        for name, factory in models.items():\n            result = evaluate_fold(values, cutoff, start, end, factory)\n            report[\'folds\'][fold][name] = result\n            print(f\'{fold} {name}: WAPE-score={result["metrics"]["all"]["wape_score"]:.6f}\', flush=True)\n        dump(args.out / \'report.json\', report)\n    proposed, losses = select_dev(report[\'folds\'], models)\n    baseline, _ = select_dev(report[\'folds\'], [\'mean_zero\', \'median_zero\'])\n    report[\'selection\'].update(development_candidate=proposed, development_baseline=baseline,\n                               development_absolute_errors=losses)\n    # Preselected candidate plus controls and calendar profile see Sep–Oct.\n    report[\'folds\'][\'sep_oct\'] = {}\n    candidates_to_eval = dict.fromkeys([baseline, \'mean_zero\', \'median_zero\', \'profile_calendar\', proposed])\n    for name in candidates_to_eval:\n        result = evaluate_fold(values, *FOLDS[2][1:], models[name])\n        report[\'folds\'][\'sep_oct\'][name] = result\n        print(f\'sep_oct {name}: WAPE-score={result["metrics"]["all"]["wape_score"]:.6f}\', flush=True)\n    latest = report[\'folds\'][\'sep_oct\']\n    err = lambda name: latest[name][\'metrics\'][\'all\'][\'absolute_error_sum\']\n    final_baseline = min([\'mean_zero\', \'median_zero\'], key=lambda n: (err(n), n))\n    best_candidate = min([c for c in [proposed, \'profile_calendar\'] if c in latest], key=lambda n: (err(n), n))\n    selected = best_candidate if err(best_candidate) < err(final_baseline) else final_baseline\n    report[\'selection\'].update(selected=selected,\n        final_baseline=final_baseline,\n        rule=\'candidate from pooled May–August error; keep it only if better than both registered baselines in Sep–Oct\',\n        caveat=\'Sep–Oct is a validation gate used in selection, not an untouched test estimate\')\n    # Refit final model on all available history through Oct 31, independently of backtest fits.\n    model = models[selected]().fit(values, FIRST, FINAL_CUTOFF)\n    predictions = model.predict(sample_keys)\n    cold = sorted({k[0] for k in sample_keys} - {k[0] for k in values}, key=int)\n    report[\'cold_start\'] = {\'routes\': cold, \'policy\': \'zero; no evidence to calibrate unseen routes\',\n                            \'evaluated\': False, \'route5_rows_preserved\': True}\n    identity = json.dumps({\'inputs\': inputs, \'code\': code, \'selected\': selected,\n                           \'parameters\': report[\'parameters\'], \'libraries\': report[\'library_versions\'],\n                           \'python\': report[\'python_version\']}, sort_keys=True).encode()\n    version = f\'ds1-{selected}-\' + hashlib.sha256(identity).hexdigest()[:12]\n    if args.synthetic:\n        version = \'synthetic-NOT-FOR-SUBMISSION-\' + version\n    report[\'model_version\'] = version\n    # Local artifact only. Never load an untrusted pickle and never publish organizer-derived artifacts.\n    artifact_path = args.out / \'model.local.pkl\'\n    artifact_path.write_bytes(pickle.dumps(model, protocol=5))\n    output_path = args.out / \'submission.csv\'\n    report[\'submission\'] = write_submission(args.sample, predictions, output_path)\n    export_api(sample_keys, predictions, args.out / \'forecast_api.csv\', version)\n    report[\'outputs_sha256\'] = {p.name: sha256(p) for p in\n                                (artifact_path, output_path, args.out / \'forecast_api.csv\')}\n    if sha256(args.sample) != inputs[\'sample\'][\'sha256\']:\n        raise RuntimeError(\'Original sample changed during run\')\n    validate(args.sample, output_path)\n    dump(args.out / \'report.json\', report)\n    print(f\'Selected {version}; validated {len(sample_keys)} rows. Platform result unknown.\', flush=True)\n    return report\n\n\nif __name__ == \'__main__\':\n    main()\n', 'ml/forecast/prepare_inputs.py': '"""Extract only labels/sample/description from a local organizer ZIP, never raw events."""\nimport argparse\nimport json\nimport zipfile\nfrom pathlib import Path\n\nfrom .core import sha256\n\n\ndef extract(archive, output):\n    required = {\'labels_day_train.csv\', \'labels_day_test.csv\', \'test_submission.csv\'}\n    output = Path(output)\n    with zipfile.ZipFile(archive) as z:\n        members = {}\n        for info in z.infolist():\n            name = Path(info.filename).name\n            if name not in required and name.lower() != \'readme.md\':\n                continue\n            if name in members:\n                raise ValueError(f\'Ambiguous archive: duplicate {name}\')\n            if info.file_size > 20_000_000:\n                raise ValueError(f\'Unexpectedly large label/description file: {name}\')\n            members[name] = info\n        if not required <= members.keys():\n            raise ValueError(f\'Missing archive members: {sorted(required - members.keys())}\')\n        if any((output / name).exists() for name in members):\n            raise ValueError(\'Refuse to overwrite original inputs; use an empty directory\')\n        output.mkdir(parents=True, exist_ok=True)\n        for name, info in members.items():\n            (output / name).write_bytes(z.read(info))\n    manifest = {\'archive_name\': Path(archive).name,\n                \'source_url\': \'https://disk.yandex.ru/d/DiFwlfMOauxjBg\',\n                \'files_sha256\': {name: sha256(output / name) for name in sorted(members)}}\n    (output / \'inputs_manifest.json\').write_text(json.dumps(manifest, indent=2) + \'\\n\')\n    return manifest\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--archive\', required=True, type=Path)\n    p.add_argument(\'--out\', required=True, type=Path)\n    a = p.parse_args()\n    print(json.dumps(extract(a.archive, a.out), indent=2))\n', 'ml/forecast/test_pipeline.py': '"""Self-contained synthetic contract/leakage tests; no organizer data required."""\nimport csv\nimport importlib.util\nimport tempfile\nimport unittest\nimport zipfile\nfrom datetime import date, timedelta\nfrom pathlib import Path\n\nfrom ml.forecast.core import grid, load_labels, rounded, score, sha256\nfrom ml.forecast.models import Boosting, Profile\nfrom ml.forecast.run import evaluate_fold, select_dev\nfrom ml.forecast.prepare_inputs import extract\nfrom ml.submit.adapter import export_api, read_sample, validate, write_submission\n\n\nclass PipelineTests(unittest.TestCase):\n    def setUp(self):\n        self.tmp = tempfile.TemporaryDirectory()\n        self.root = Path(self.tmp.name)\n\n    def tearDown(self):\n        self.tmp.cleanup()\n\n    def sample(self):\n        path = self.root / \'sample.csv\'\n        # Deliberately reverse order to detect implicit model sorting of sample rows.\n        keys = grid([str(i) for i in range(1, 11)], date(2025, 11, 1), date(2025, 12, 31))[::-1]\n        with path.open(\'w\', newline=\'\') as f:\n            writer = csv.writer(f, delimiter=\';\')\n            writer.writerow([\'route\', \'date\', \'hour\', \'prediction\'])\n            writer.writerows((r, d, h, \'\') for r, d, h in keys)\n        return path, keys\n\n    def test_exact_sample_roundtrip_api_sums_and_route5(self):\n        sample, keys = self.sample()\n        digest = sha256(sample)\n        preds = {k: (int(k[0]) * 10 + k[2] + .5) for k in keys}\n        out = self.root / \'out.csv\'\n        result = write_submission(sample, preds, out)\n        self.assertEqual(result[\'route5_rows\'], 1464)\n        self.assertEqual(read_sample(out)[1], keys)\n        self.assertEqual(sha256(sample), digest)\n        api = self.root / \'api.csv\'\n        export_api(keys, preds, api, \'synthetic-test\')\n        with api.open() as f:\n            rows = list(csv.DictReader(f))\n        self.assertEqual(sum(int(r[\'pred_validations\']) for r in rows), result[\'prediction_sum\'])\n        self.assertTrue(all(r[\'timestamp_msk\'].endswith(\'+03:00\') for r in rows))\n        daily = {}\n        monthly = {}\n        for row in rows:\n            d = row[\'timestamp_msk\'][:10]\n            daily[d] = daily.get(d, 0) + int(row[\'pred_validations\'])\n        for d, value in daily.items():\n            monthly[d[:7]] = monthly.get(d[:7], 0) + value\n        self.assertEqual(sum(monthly.values()), result[\'prediction_sum\'])\n\n    def test_reject_bad_submission_inputs(self):\n        sample, keys = self.sample()\n        predictions = dict.fromkeys(keys, 1)\n        out = self.root / \'out.csv\'\n        with self.assertRaises(ValueError):\n            write_submission(sample, predictions, sample)\n        predictions.pop(keys[0])\n        with self.assertRaises(ValueError):\n            write_submission(sample, predictions, out)\n        predictions[keys[0]] = float(\'nan\')\n        with self.assertRaises(ValueError):\n            write_submission(sample, predictions, out)\n        predictions[keys[0]] = 1\n        write_submission(sample, predictions, out)\n        lines = out.read_text().splitlines()\n        lines[1], lines[2] = lines[2], lines[1]\n        out.write_text(\'\\n\'.join(lines) + \'\\n\')\n        with self.assertRaises(ValueError):\n            validate(sample, out)\n\n    def test_sample_missing_duplicate_wrong_period(self):\n        path, _ = self.sample()\n        lines = path.read_text().splitlines()\n        path.write_text(\'\\n\'.join(lines[:-1]) + \'\\n\')\n        with self.assertRaises(ValueError):\n            read_sample(path)\n        path.write_text(\'\\n\'.join(lines[:-1] + [lines[1]]) + \'\\n\')\n        with self.assertRaises(ValueError):\n            read_sample(path)\n        path.write_text(\'\\n\'.join(lines).replace(\'2025-11-01\', \'2025-10-31\') + \'\\n\')\n        with self.assertRaises(ValueError):\n            read_sample(path)\n\n    def test_metric_is_pooled_and_includes_missing_hours(self):\n        keys = [(\'1\', date(2025, 9, 1), 7), (\'2\', date(2025, 9, 1), 7)]\n        m = score(keys, {keys[0]: 100}, {keys[0]: 80, keys[1]: 10})\n        self.assertAlmostEqual(m[\'all\'][\'wape_score\'], .7)\n        self.assertIsNone(m[\'route:2\'][\'wape_score\'])\n        with self.assertRaises(ValueError):\n            score(keys, {}, dict.fromkeys(keys, 0))\n        with self.assertRaises(ValueError):\n            score(keys, {keys[0]: 1}, {keys[0]: 1})\n\n    def test_reader_rejects_duplicate_files_and_negative_counts(self):\n        p = self.root / \'labels.csv\'\n        p.write_text(\'route;date;hour;boardings\\n7;2025-01-01;0;2\\n\')\n        with self.assertRaises(ValueError):\n            load_labels([p, p])\n        p.write_text(\'route;date;hour;boardings\\n7;2025-01-01;0;-2\\n\')\n        with self.assertRaises(ValueError):\n            load_labels([p])\n\n    def test_fit_rejects_future_and_predict_rejects_past(self):\n        cutoff = date(2025, 1, 31)\n        history = {(\'7\', date(2025, 1, 1), 7): 20}\n        model = Profile().fit(history, date(2025, 1, 1), cutoff)\n        with self.assertRaises(ValueError):\n            model.predict(list(history))\n        history[\'7\', cutoff + timedelta(days=1), 7] = 1000\n        with self.assertRaises(ValueError):\n            Profile().fit(history, date(2025, 1, 1), cutoff)\n\n    def test_backtest_never_passes_future_labels_to_model(self):\n        values = {(\'7\', date(2025, 1, 1), 7): 10,\n                  (\'7\', date(2025, 2, 1), 7): 2000}\n        class Spy(Profile):\n            def fit(self, history, start, cutoff):\n                assert max(k[1] for k in history) <= cutoff\n                assert max(history.values()) == 10\n                return super().fit(history, start, cutoff)\n        result = evaluate_fold(values, date(2025, 1, 31), date(2025, 2, 1), date(2025, 2, 1), Spy)\n        self.assertEqual(result[\'metrics\'][\'all\'][\'rows\'], 24)\n\n    def test_zero_cold_start_and_missing_training_sensitivity(self):\n        history = {(\'7\', date(2025, 1, 1), 7): 40}\n        keys = [(\'7\', date(2025, 2, 5), 7), (\'5\', date(2025, 2, 5), 7)]\n        zero = Profile().fit(history, date(2025, 1, 1), date(2025, 1, 31)).predict(keys)\n        obs = Profile(missing=\'observed\').fit(history, date(2025, 1, 1), date(2025, 1, 31)).predict(keys)\n        self.assertEqual(zero[keys[1]], 0)\n        self.assertLess(zero[keys[0]], obs[keys[0]])\n\n    def test_rounding_is_finite_nonnegative(self):\n        self.assertEqual([rounded(x) for x in [-2, .5, 1.5, 2.5]], [0, 0, 2, 2])\n        for x in [float(\'inf\'), float(\'nan\'), 10**15]:\n            with self.assertRaises(ValueError):\n                rounded(x)\n\n    def test_selection_does_not_use_sep_oct(self):\n        def entry(n):\n            return {\'metrics\': {\'all\': {\'absolute_error_sum\': n}}}\n        folds = {\'may_jun\': {\'a\': entry(1), \'b\': entry(10)},\n                 \'jul_aug\': {\'a\': entry(5), \'b\': entry(1)},\n                 \'sep_oct\': {\'a\': entry(10000), \'b\': entry(0)}}\n        self.assertEqual(select_dev(folds, [\'a\', \'b\'])[0], \'a\')\n\n    def test_archive_extracts_only_needed_files_and_preserves_originals(self):\n        archive = self.root / \'data.zip\'\n        with zipfile.ZipFile(archive, \'w\') as z:\n            for name in [\'labels_day_train.csv\', \'labels_day_test.csv\', \'test_submission.csv\']:\n                z.writestr(\'nested/\' + name, \'synthetic\')\n            z.writestr(\'../../raw.csv\', \'must not be extracted\')\n            z.writestr(\'README.md\', \'Synthetic description\')\n        dest = self.root / \'inputs\'\n        manifest = extract(archive, dest)\n        self.assertEqual(len(manifest[\'files_sha256\']), 4)\n        self.assertFalse((self.root / \'raw.csv\').exists())\n        self.assertFalse((dest / \'raw.csv\').exists())\n        with self.assertRaises(ValueError):\n            extract(archive, dest)\n\n    @unittest.skipUnless(importlib.util.find_spec(\'numpy\'), \'optional boosting dependency\')\n    def test_recursive_lags_use_predictions_and_fill_intervening_days(self):\n        class FakeRegressor:\n            def __init__(self):\n                self.batches = []\n            def predict(self, features):\n                self.batches.append(features)\n                return [7] * len(features)\n        model = Boosting(recursive=True)\n        model.cutoff = date(2025, 10, 31)\n        model.routes, model.route_codes = [\'7\'], {\'7\': 0}\n        model.history = {(\'7\', model.cutoff, h): 99 for h in range(24)}\n        model.model = FakeRegressor()\n        key = (\'7\', date(2025, 11, 3), 12)\n        self.assertEqual(model.predict([key]), {key: 7})\n        self.assertEqual(len(model.model.batches), 3)\n        self.assertTrue(all(row[8] == 99 for row in model.model.batches[0]))\n        self.assertTrue(all(row[8] == 7 for row in model.model.batches[1]))\n        self.assertTrue(all(row[8] == 7 for row in model.model.batches[2]))\n        self.assertEqual(model.history[\'7\', model.cutoff, 12], 99)\n\n    def test_russian_calendar_2025(self):\n        from ml.forecast.core import is_workday, effective_weekday, calendar_day_type\n        # 2025-11-01 is a working Saturday\n        self.assertTrue(is_workday(date(2025, 11, 1)))\n        self.assertEqual(effective_weekday(date(2025, 11, 1)), 4)\n        self.assertEqual(calendar_day_type(date(2025, 11, 1)), 4)\n\n        # 2025-11-02 is Sunday\n        self.assertFalse(is_workday(date(2025, 11, 2)))\n        self.assertEqual(effective_weekday(date(2025, 11, 2)), 6)\n\n        # 2025-11-03 is non-working day (transferred holiday)\n        self.assertFalse(is_workday(date(2025, 11, 3)))\n        self.assertEqual(effective_weekday(date(2025, 11, 3)), 6)\n\n        # 2025-11-04 is National Unity Day\n        self.assertFalse(is_workday(date(2025, 11, 4)))\n        self.assertEqual(effective_weekday(date(2025, 11, 4)), 6)\n\n        # 2025-11-05 is normal Wednesday\n        self.assertTrue(is_workday(date(2025, 11, 5)))\n        self.assertEqual(effective_weekday(date(2025, 11, 5)), 2)\n\n        # 2025-12-31 is non-working day\n        self.assertFalse(is_workday(date(2025, 12, 31)))\n        self.assertEqual(effective_weekday(date(2025, 12, 31)), 6)\n\n    def test_calendar_profile_and_route50_rule(self):\n        from ml.forecast.models import CalendarProfile\n        history = {}\n        for d in [date(2025, 10, 1) + timedelta(days=i) for i in range(30)]:\n            for h in range(24):\n                history[\'50\', d, h] = 50 if d.weekday() < 5 else 0\n                history[\'1\', d, h] = 100 if d.weekday() < 5 else 20\n        cutoff = date(2025, 10, 31)\n        model = CalendarProfile(recent_weight=0.5, window=28, route50_rule=True).fit(history, date(2025, 10, 1), cutoff)\n\n        # 2025-11-01 is working Saturday: route 1 should have weekday traffic (~100)\n        p1_work = model.predict([(\'1\', date(2025, 11, 1), 12)])[(\'1\', date(2025, 11, 1), 12)]\n        self.assertGreater(p1_work, 80)\n\n        # 2025-11-04 is holiday: route 1 should have weekend/Sunday traffic (~20)\n        p1_hol = model.predict([(\'1\', date(2025, 11, 4), 12)])[(\'1\', date(2025, 11, 4), 12)]\n        self.assertLess(p1_hol, 30)\n\n        # Route 50 suspended on weekend before Nov 15\n        p50_susp = model.predict([(\'50\', date(2025, 11, 2), 12)])[(\'50\', date(2025, 11, 2), 12)]\n        self.assertEqual(p50_susp, 0)\n\n    @unittest.skipUnless(importlib.util.find_spec(\'lightgbm\'), \'lightgbm required\')\n    def test_lgbm_direct_fit_and_predict(self):\n        history = {}\n        for d in [date(2025, 9, 1) + timedelta(days=i) for i in range(30)]:\n            for h in range(24):\n                history[\'1\', d, h] = 100 + h * 2 if d.weekday() < 5 else 20 + h\n        cutoff = date(2025, 9, 30)\n        model = Boosting(kind=\'lgb\', recursive=False).fit(history, date(2025, 9, 1), cutoff)\n        preds = model.predict([(\'1\', date(2025, 10, 1), 10), (\'1\', date(2025, 10, 4), 10)])\n        self.assertGreater(preds[\'1\', date(2025, 10, 1), 10], preds[\'1\', date(2025, 10, 4), 10])\n\n    @unittest.skipUnless(importlib.util.find_spec(\'catboost\'), \'catboost required\')\n    def test_catboost_direct_fit_and_predict(self):\n        history = {}\n        for d in [date(2025, 9, 1) + timedelta(days=i) for i in range(30)]:\n            for h in range(24):\n                history[\'1\', d, h] = 100 + h * 2 if d.weekday() < 5 else 20 + h\n        cutoff = date(2025, 9, 30)\n        model = Boosting(kind=\'catboost\', recursive=False).fit(history, date(2025, 9, 1), cutoff)\n        preds = model.predict([(\'1\', date(2025, 10, 1), 10)])\n        self.assertGreater(preds[\'1\', date(2025, 10, 1), 10], 50)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n\n', 'ml/forecast/download_yandex.py': '"""Download organizer ZIP through the public Yandex Disk API, without credentials."""\nimport hashlib\nimport json\nimport re\nimport shutil\nimport time\nimport urllib.parse\nimport urllib.request\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nPUBLIC_URL = \'https://disk.yandex.ru/d/DiFwlfMOauxjBg\'\nAPI = \'https://cloud-api.yandex.net/v1/disk/public/resources\'\n\n\ndef api_json(suffix, params, attempts=3):\n    url = API + suffix + \'?\' + urllib.parse.urlencode(params)\n    for attempt in range(attempts):\n        try:\n            with urllib.request.urlopen(url, timeout=30) as response:\n                return json.load(response)\n        except (OSError, ValueError) as exc:\n            if attempt == attempts - 1:\n                raise RuntimeError(\'API Яндекс Диска недоступен. Включите Internet в Kaggle; \'\n                                   \'если уже включён, повторите ячейку позже. \'\n                                   \'Также возможен лимит скачивания публичной ссылки.\') from exc\n            print(f\'API: повторная попытка {attempt + 2}/{attempts}\', flush=True)\n            time.sleep(2 ** (attempt + 1))\n\n\ndef digest(path, algorithm=\'sha256\'):\n    with Path(path).open(\'rb\') as f:\n        return hashlib.file_digest(f, algorithm).hexdigest()\n\n\ndef download(public_url=PUBLIC_URL, cache_dir=\'/tmp/mthack_yandex\', attempts=3):\n    params = {\'public_key\': public_url}\n    meta = api_json(\'\', params)\n    if meta.get(\'type\') == \'dir\':\n        # Organizer link is a public folder containing dataset.zip.\n        params[\'path\'] = \'/dataset.zip\'\n        meta = api_json(\'\', params)\n    if meta.get(\'type\') != \'file\' or not meta.get(\'name\', \'\').lower().endswith(\'.zip\'):\n        raise ValueError(\'Ссылка должна вести на ZIP или папку с dataset.zip\')\n    expected_size = int(meta[\'size\'])\n    if expected_size <= 0:\n        raise ValueError(\'Пустой архив в метаданных Яндекс Диска\')\n    algorithm = \'sha256\' if meta.get(\'sha256\') else \'md5\'\n    expected_hash = meta.get(algorithm)\n    identity = hashlib.sha256(json.dumps(params, sort_keys=True).encode()).hexdigest()[:16]\n    directory = Path(cache_dir) / identity\n    directory.mkdir(parents=True, exist_ok=True)\n    target, partial = directory / \'dataset.zip\', directory / \'dataset.zip.part\'\n\n    def valid(path):\n        return (path.exists() and path.stat().st_size == expected_size\n                and (not expected_hash or digest(path, algorithm) == expected_hash)\n                and zipfile.is_zipfile(path))\n\n    if valid(target):\n        print(\'Используется ранее проверенный архив:\', target, flush=True)\n        return target\n    print(f\'Архив: {expected_size / 1024**3:.2f} ГиБ. Потоковое скачивание во временную папку.\',\n          flush=True)\n    for attempt in range(attempts):\n        try:\n            offset = partial.stat().st_size if partial.exists() else 0\n            if offset >= expected_size:\n                if valid(partial):\n                    break\n                partial.unlink()\n                offset = 0\n            if shutil.disk_usage(directory).free < expected_size - offset + 64 * 1024**2:\n                raise RuntimeError(\'Недостаточно места для архива во временной папке\')\n            href = api_json(\'/download\', params)[\'href\']\n            if urllib.parse.urlparse(href).scheme != \'https\':\n                raise ValueError(\'API вернул незащищённую ссылку скачивания\')\n            headers = {\'Range\': f\'bytes={offset}-\'} if offset else {}\n            request = urllib.request.Request(href, headers=headers)\n            with urllib.request.urlopen(request, timeout=60) as response:\n                status = response.status\n                if status == 206:\n                    match = re.fullmatch(r\'bytes (\\d+)-(\\d+)/(\\d+)\',\n                                         response.headers.get(\'Content-Range\', \'\'))\n                    if not match or int(match[1]) != offset or int(match[3]) != expected_size:\n                        raise ValueError(\'Некорректный диапазон при докачке\')\n                    mode = \'ab\'\n                elif status == 200:\n                    offset, mode = 0, \'wb\'  # Server ignored Range: safely restart.\n                else:\n                    raise ValueError(f\'Неожиданный HTTP-статус: {status}\')\n                received, logged = offset, offset\n                with partial.open(mode) as f:\n                    while chunk := response.read(4 * 1024**2):\n                        f.write(chunk)\n                        received += len(chunk)\n                        if received > expected_size:\n                            raise ValueError(\'Размер загрузки превысил метаданные\')\n                        if received - logged >= 64 * 1024**2 or received == expected_size:\n                            print(f\'Скачано {received / expected_size:.1%} \'\n                                  f\'({received / 1024**2:.0f}/{expected_size / 1024**2:.0f} МиБ)\',\n                                  flush=True)\n                            logged = received\n            if partial.stat().st_size != expected_size:\n                raise OSError(\'Соединение прервано до конца файла\')\n            if not valid(partial):\n                partial.unlink()\n                raise ValueError(\'Проверка хеша или структуры ZIP не пройдена\')\n            break\n        except (OSError, ValueError) as exc:\n            if attempt == attempts - 1:\n                raise RuntimeError(\'Архив не скачан после повторных попыток. \'\n                                   \'Проверьте Internet и повторите ячейку: частичный файл сохранён \'\n                                   \'для докачки, если сервер поддерживает Range.\') from exc\n            print(f\'Скачивание: повторная попытка {attempt + 2}/{attempts}\', flush=True)\n            time.sleep(2 ** (attempt + 1))\n    partial.replace(target)\n    manifest = {\'source_url\': public_url, \'path\': params.get(\'path\'), \'size\': expected_size,\n                \'sha256\': digest(target), \'fetched_at_utc\': datetime.now(timezone.utc).isoformat(),\n                \'server_hash_algorithm\': algorithm if expected_hash else None,\n                \'server_hash_verified\': bool(expected_hash)}\n    (directory / \'download_manifest.json\').write_text(json.dumps(manifest, indent=2) + \'\\n\')\n    print(\'Архив скачан и проверен:\', target, flush=True)\n    return target\n', 'ml/forecast/test_download_yandex.py': '"""Network-independent tests: streaming, resume, cache and integrity checks."""\nimport hashlib\nimport io\nimport tempfile\nimport unittest\nimport zipfile\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nfrom ml.forecast import download_yandex as yd\n\n\nclass Response(io.BytesIO):\n    def __init__(self, data, status=200, headers=None):\n        super().__init__(data)\n        self.status = status\n        self.headers = headers or {}\n\n\nclass DownloadTests(unittest.TestCase):\n    def setUp(self):\n        self.tmp = tempfile.TemporaryDirectory()\n        self.root = Path(self.tmp.name)\n        buffer = io.BytesIO()\n        with zipfile.ZipFile(buffer, \'w\') as z:\n            z.writestr(\'labels/synthetic.txt\', \'artificial fixture\')\n        self.blob = buffer.getvalue()\n        self.meta = {\'type\': \'file\', \'name\': \'dataset.zip\', \'size\': len(self.blob),\n                     \'sha256\': hashlib.sha256(self.blob).hexdigest()}\n\n    def tearDown(self):\n        self.tmp.cleanup()\n\n    def api(self, suffix, params):\n        if suffix == \'/download\':\n            self.assertEqual(params[\'path\'], \'/dataset.zip\')\n            return {\'href\': \'https://download.example.invalid/data.zip\'}\n        return self.meta if \'path\' in params else {\'type\': \'dir\'}\n\n    def run_download(self, opener, attempts=2):\n        with patch.object(yd, \'api_json\', side_effect=self.api), \\\n             patch.object(yd.urllib.request, \'urlopen\', side_effect=opener), \\\n             patch.object(yd.time, \'sleep\'):\n            return yd.download(cache_dir=self.root, attempts=attempts)\n\n    def test_folder_download_and_verified_cache(self):\n        target = self.run_download(lambda *a, **kw: Response(self.blob))\n        self.assertEqual(target.read_bytes(), self.blob)\n        with patch.object(yd, \'api_json\', side_effect=self.api), \\\n             patch.object(yd.urllib.request, \'urlopen\') as opened:\n            self.assertEqual(yd.download(cache_dir=self.root), target)\n            opened.assert_not_called()\n\n    def test_interruption_resumes_with_range(self):\n        blob = self.blob\n        class Interrupted(Response):\n            def read(self, size=-1):\n                if self.tell():\n                    raise OSError(\'simulated disconnect\')\n                return super().read(40)\n        calls = []\n        def open_response(request, **kwargs):\n            calls.append(request)\n            if len(calls) == 1:\n                return Interrupted(blob)\n            self.assertEqual(request.get_header(\'Range\'), \'bytes=40-\')\n            return Response(blob[40:], 206,\n                            {\'Content-Range\': f\'bytes 40-{len(blob)-1}/{len(blob)}\'})\n        self.assertEqual(self.run_download(open_response).read_bytes(), self.blob)\n        self.assertEqual(len(calls), 2)\n\n    def test_ignored_range_restarts_instead_of_appending(self):\n        class Interrupted(Response):\n            def read(self, size=-1):\n                if self.tell():\n                    raise OSError(\'disconnect\')\n                return super().read(40)\n        responses = iter([Interrupted(self.blob), Response(self.blob)])\n        self.assertEqual(self.run_download(lambda *a, **kw: next(responses)).read_bytes(), self.blob)\n\n    def test_wrong_hash_never_becomes_completed_archive(self):\n        self.meta[\'sha256\'] = \'0\' * 64\n        with self.assertRaises(RuntimeError):\n            self.run_download(lambda *a, **kw: Response(self.blob), attempts=1)\n        self.assertFalse(list(self.root.rglob(\'dataset.zip\')))\n\n    def test_wrong_range_never_becomes_completed_archive(self):\n        with self.assertRaises(RuntimeError):\n            self.run_download(lambda *a, **kw: Response(self.blob, 206,\n                              {\'Content-Range\': f\'bytes 5-{len(self.blob)-1}/{len(self.blob)}\'}),\n                              attempts=1)\n        self.assertFalse(list(self.root.rglob(\'dataset.zip\')))\n\n    def test_api_retries_and_reports_network_problem(self):\n        with patch.object(yd.urllib.request, \'urlopen\', side_effect=OSError(\'timeout\')) as opened, \\\n             patch.object(yd.time, \'sleep\'):\n            with self.assertRaisesRegex(RuntimeError, \'Internet\'):\n                yd.api_json(\'\', {\'public_key\': yd.PUBLIC_URL}, attempts=2)\n            self.assertEqual(opened.call_count, 2)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'ml/submit/adapter.py': '"""Preserve the exact organizer sample keys, ordering and non-target text."""\nfrom __future__ import annotations\n\nimport csv\nfrom datetime import date, datetime\nfrom pathlib import Path\nfrom zoneinfo import ZoneInfo\n\nfrom ml.forecast.core import grid, parse_key, rounded\n\nFIELDS = [\'route\', \'date\', \'hour\', \'prediction\']\nSTART, END = date(2025, 11, 1), date(2025, 12, 31)\n\n\ndef read_sample(path):\n    with Path(path).open(encoding=\'utf-8-sig\', newline=\'\') as f:\n        reader = csv.DictReader(f, delimiter=\';\')\n        if reader.fieldnames != FIELDS:\n            raise ValueError(f\'Unexpected sample schema: {reader.fieldnames}\')\n        rows = list(reader)\n    if any(None in r or any(v is None for v in r.values()) for r in rows):\n        raise ValueError(\'Malformed sample row\')\n    keys = [parse_key(r) for r in rows]\n    routes = sorted({k[0] for k in keys}, key=int)\n    if len(keys) != 14640 or len(set(keys)) != len(keys):\n        raise ValueError(\'Sample must contain exactly 14640 unique keys\')\n    if len(routes) != 10 or \'5\' not in routes:\n        raise ValueError(\'Sample must contain ten routes, including route 5\')\n    if set(keys) != set(grid(routes, START, END)):\n        raise ValueError(\'Sample must cover every hour of November–December 2025\')\n    return rows, keys\n\n\ndef validate(sample_path, output_path):\n    rows, keys = read_sample(sample_path)\n    other, other_keys = read_sample(output_path)\n    if keys != other_keys:\n        raise ValueError(\'Submission changed sample ordering\')\n    for expected, actual in zip(rows, other):\n        if any(expected[k] != actual[k] for k in FIELDS[:-1]):\n            raise ValueError(\'Non-target sample text changed\')\n        value = actual[\'prediction\']\n        if not value.isascii() or not value.isdecimal() or str(int(value)) != value:\n            raise ValueError(\'Submission predictions must be canonical nonnegative integers\')\n        rounded(int(value))\n    return {\'rows\': len(keys), \'routes\': len({k[0] for k in keys}),\n            \'route5_rows\': sum(k[0] == \'5\' for k in keys), \'prediction_sum\':\n            sum(int(row[\'prediction\']) for row in other)}\n\n\ndef write_submission(sample_path, predictions, output_path):\n    if Path(sample_path).resolve() == Path(output_path).resolve():\n        raise ValueError(\'Never overwrite the original sample\')\n    rows, keys = read_sample(sample_path)\n    if set(predictions) != set(keys):\n        raise ValueError(\'Prediction keys must match sample exactly; no missing/extra rows\')\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    values = [rounded(predictions[k]) for k in keys]\n    with output_path.open(\'w\', encoding=\'utf-8\', newline=\'\') as f:\n        writer = csv.DictWriter(f, fieldnames=FIELDS, delimiter=\';\', lineterminator=\'\\n\')\n        writer.writeheader()\n        for row, value in zip(rows, values):\n            writer.writerow({**row, \'prediction\': value})\n    return validate(sample_path, output_path)\n\n\ndef export_api(keys, predictions, output_path, model_version):\n    if len(set(keys)) != len(keys) or set(keys) != set(predictions):\n        raise ValueError(\'API export key mismatch\')\n    with Path(output_path).open(\'w\', encoding=\'utf-8\', newline=\'\') as f:\n        writer = csv.writer(f, lineterminator=\'\\n\')\n        writer.writerow([\'route_id\', \'timestamp_msk\', \'pred_validations\',\n                         \'model_version\', \'source_mode\'])\n        for r, d, h in keys:\n            stamp = datetime(d.year, d.month, d.day, h, tzinfo=ZoneInfo(\'Europe/Moscow\'))\n            writer.writerow([r, stamp.isoformat(), rounded(predictions[r, d, h]),\n                             model_version, \'as_of_2025_10_31\'])\n\n\nif __name__ == \'__main__\':\n    import argparse\n    import json\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--sample\', required=True)\n    parser.add_argument(\'--submission\', required=True)\n    args = parser.parse_args()\n    print(json.dumps(validate(args.sample, args.submission), ensure_ascii=False, indent=2))\n', 'ml/forecast/weather_competition.py': '"""Paired weather ablation. Competition mode uses allowed retrospective weather.\n\nNever consumes future passenger counts. The frozen reference was selected in\nprior Sep-Oct research, so those scores are exploratory, not independent tests.\n"""\nfrom __future__ import annotations\nimport argparse\nimport calendar\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport pickle\nfrom datetime import date,datetime,timedelta,timezone\nfrom pathlib import Path\nimport numpy as np\nfrom .core import grid,load_labels,rounded,score,sha256,effective_weekday,calendar_day_type\nfrom .route_specific import verify_inputs,candidate_factories,RouteEnsemble\nfrom .run import FIRST,FINAL_CUTOFF,FOLDS,dump\nfrom ml.submit.adapter import read_sample,write_submission\n\nVARIABLES=(\'temperature_2m\',\'relative_humidity_2m\',\'precipitation\',\'rain\',\n           \'snowfall\',\'wind_speed_10m\',\'cloud_cover\')\nWEIGHTS=(.25,.5,1.)\n\n\ndef weather_rows(payload):\n    if payload.get(\'timezone\')!=\'Europe/Moscow\' or payload.get(\'utc_offset_seconds\')!=10800:\n        raise ValueError(\'Weather must use Europe/Moscow (+03:00)\')\n    h=payload[\'hourly\'];times=h[\'time\']\n    if len(set(times))!=len(times):raise ValueError(\'Duplicate weather timestamp\')\n    if any(len(h[v])!=len(times) for v in VARIABLES):raise ValueError(\'Weather lengths differ\')\n    result={};previous=None\n    for i,stamp in enumerate(times):\n        t=datetime.fromisoformat(stamp)\n        if t.minute or t.second or (previous and t-previous!=timedelta(hours=1)):\n            raise ValueError(\'Weather hours are not contiguous\')\n        previous=t\n        raw=[h[v][i] for v in VARIABLES]\n        if any(v is None or not math.isfinite(float(v)) for v in raw):\n            raise ValueError(f\'Missing/nonfinite weather at {stamp}; do not silently impute\')\n        raw=list(map(float,raw))\n        # Only preceding/current weather hours in rolling summaries.\n        j=max(0,i-23)\n        raw.extend([float(np.mean(h[\'temperature_2m\'][j:i+1])),\n                    float(np.sum(h[\'precipitation\'][j:i+1])),float(raw[0]<0)])\n        result[t.date(),t.hour]=raw\n    return result\n\n\ndef fit_reference(history,cutoff,choices):\n    factories=candidate_factories(profiles_only=True)\n    names={\'school_calendar\'}|{c[\'candidate\'] for c in choices.values()}\n    models={n:factories[n]().fit(history,FIRST,cutoff) for n in names}\n    return RouteEnsemble(models,choices,cutoff)\n\n\ndef matrix(keys,base,weather,use_weather):\n    rows=[]\n    for k in keys:\n        r,d,h=k\n        row=[int(r),h,d.weekday(),effective_weekday(d),calendar_day_type(d),\n             int(d.month in (7,8)),base[k],math.sin(2*math.pi*h/24),math.cos(2*math.pi*h/24)]\n        if use_weather:\n            if (d,h) not in weather:raise ValueError(f\'Weather missing for {d} {h}\')\n            row.extend(weather[d,h])\n        rows.append(row)\n    return np.asarray(rows,dtype=float)\n\n\ndef causal_residual_training(history,cutoff,choices):\n    """Monthly out-of-time reference predictions; no same-row target encoding."""\n    routes=sorted({k[0] for k in history},key=int)\n    keys=[];base={};blocks=[]\n    lo=date(2025,2,1)\n    while lo<=cutoff:\n        hi=min(cutoff,date(lo.year,lo.month,calendar.monthrange(lo.year,lo.month)[1]))\n        training_cutoff=lo-timedelta(days=1)\n        prefix={k:y for k,y in history.items() if k[1]<=training_cutoff}\n        model=fit_reference(prefix,training_cutoff,choices)\n        batch=grid(routes,lo,hi)\n        base.update(model.predict(batch));keys.extend(batch)\n        blocks.append({\'train_through\':str(training_cutoff),\'predict_from\':str(lo),\'predict_to\':str(hi)})\n        lo=hi+timedelta(days=1)\n    if not keys:raise ValueError(\'At least two months of history required\')\n    scale=np.asarray([max(50,base[k]) for k in keys],dtype=float)\n    target=np.asarray([(history.get(k,0)-base[k])/s for k,s in zip(keys,scale)],dtype=float)\n    return keys,base,scale,target,blocks\n\n\ndef fit_pair(history,cutoff,choices,weather):\n    import lightgbm as lgb\n    keys,base,scale,y,blocks=causal_residual_training(history,cutoff,choices)\n    models={}\n    for name,use_weather in [(\'calendar\',False),(\'weather\',True)]:\n        reg=lgb.LGBMRegressor(objective=\'regression_l1\',n_estimators=220,\n            learning_rate=.025,num_leaves=12,min_child_samples=120,\n            reg_lambda=10.,max_depth=5,n_jobs=2,verbosity=-1,random_state=2025)\n        reg.fit(matrix(keys,base,weather,use_weather),y,sample_weight=scale,\n                categorical_feature=[0,1,2,3,4,5])\n        models[name]=reg\n    return models,blocks\n\n\ndef corrected(keys,base,raw,weight):\n    """Bound changes to 35% of nonzero reference; do not invent service at zero."""\n    out={}\n    for k,v in zip(keys,raw):\n        b=base[k]\n        delta=float(np.clip(float(v)*max(50,b),-.35*b,.35*b))\n        out[k]=rounded(b+weight*delta)\n    return out\n\n\ndef predictions_pair(keys,base,models,weather):\n    preds={\'reference\':base}\n    for name,use_weather in [(\'calendar\',False),(\'weather\',True)]:\n        raw=models[name].predict(matrix(keys,base,weather,use_weather))\n        for weight in WEIGHTS:preds[f\'{name}_{weight}\']=corrected(keys,base,raw,weight)\n    return preds\n\n\ndef select(keys,truth,preds,routes):\n    """Require >=0.5% gain in every validation month, with matched weather control."""\n    months=sorted({k[1].month for k in keys})\n    chosen={};control_chosen={};diagnostics={}\n    for r in routes:\n        groups=[[k for k in keys if k[0]==r and k[1].month==m] for m in months]\n        errors={name:[sum(abs(p[k]-truth.get(k,0)) for k in g) for g in groups]\n                for name,p in preds.items()}\n        wins=lambda name,ref: all(a<=.995*b for a,b in zip(errors[name],errors[ref])) and sum(errors[name])<sum(errors[ref])\n        controls=[\'reference\']+[f\'calendar_{w}\' for w in WEIGHTS if wins(f\'calendar_{w}\',\'reference\')]\n        control=min(controls,key=lambda n:(sum(errors[n]),n));control_chosen[r]=control\n        candidates=list(controls)\n        for w in WEIGHTS:\n            name=f\'weather_{w}\'\n            if wins(name,\'reference\') and wins(name,f\'calendar_{w}\') and wins(name,control):\n                candidates.append(name)\n        chosen[r]=min(candidates,key=lambda n:(sum(errors[n]),n))\n        diagnostics[r]={\'errors_by_month\':errors,\'months\':months,\'calendar_control\':control,\'selected\':chosen[r]}\n    return chosen,control_chosen,diagnostics\n\n\ndef join(keys,preds,choice):\n    return {k:preds[choice.get(k[0],\'reference\')][k] if k[0] in choice else 0 for k in keys}\n\n\nclass WeatherEnsemble:\n    def __init__(self,reference,models,weather,choices):\n        self.reference,self.models,self.weather,self.choices=reference,models,weather,choices\n    def predict(self,keys):\n        base=self.reference.predict(keys)\n        return join(keys,predictions_pair(keys,base,self.models,self.weather),self.choices)\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__)\n    for n in [\'train\',\'test\',\'sample\',\'weather\',\'reference\',\'out\']:p.add_argument(\'--\'+n,type=Path,required=True)\n    args=p.parse_args(argv)\n    inputs=verify_inputs({n:getattr(args,n) for n in [\'train\',\'test\',\'sample\']})\n    train,test=load_labels([args.train]),load_labels([args.test])\n    if any(not FIRST<=k[1]<=date(2025,8,31) for k in train):raise ValueError(\'Bad train dates\')\n    if any(not date(2025,9,1)<=k[1]<=FINAL_CUTOFF for k in test):raise ValueError(\'Bad test dates\')\n    values=load_labels([args.train,args.test]);_,sample=read_sample(args.sample)\n    ref=json.loads(args.reference.read_text());choices=ref[\'choices\']\n    routes=sorted({k[0] for k in values},key=int)\n    if set(routes)!=set(choices):raise ValueError(\'Reference route set differs from real labels\')\n    weather=weather_rows(json.loads(args.weather.read_text()))\n    for _,d,h in grid([\'weather\'],FIRST,date(2025,12,31)):\n        if (d,h) not in weather:raise ValueError(\'Weather must cover the entire 2025 calendar year\')\n    args.out.mkdir(parents=True,exist_ok=True)\n    report={\'source_mode\':\'competition\',\'external_information\':\'Retrospective ERA5 weather, allowed by organizer Q&A; not an as-of Oct31 weather forecast.\',\n            \'weather_source\':\'https://open-meteo.com/en/docs/historical-weather-api\',\n            \'weather_source_metadata\':json.loads(Path(__file__).with_name(\'weather_source.json\').read_text()),\n            \'weather_sha256\':sha256(args.weather),\'reference_sha256\':sha256(args.reference),\n            \'reference\':ref,\'inputs\':inputs,\'data_kind\':\'verified_organizer_labels\',\n            \'selection_warning\':ref[\'selection_warning\'],\'folds\':{},\'platform_score\':None,\n            \'weather_features\':list(VARIABLES)+[\'temperature_24h_mean\',\'precipitation_24h_sum\',\'below_freezing\'],\n            \'excluded_variable\':\'snow_depth is all null in downloaded ERA5 response; not used\',\n            \'correction_limit\':.35,\'weights\':list(WEIGHTS),\n            \'fitted_at\':datetime.now(timezone.utc).isoformat(),\n            \'libraries\':{n:importlib.metadata.version(n) for n in [\'numpy\',\'lightgbm\',\'scikit-learn\']},\n            \'code_sha256\':{p.name:sha256(p) for p in sorted(Path(__file__).parent.glob(\'*.py\'))}}\n    for label,cutoff,lo,hi in FOLDS:\n        history={k:y for k,y in values.items() if k[1]<=cutoff}\n        keys=grid(routes,lo,hi);truth={k:y for k,y in values.items() if lo<=k[1]<=hi}\n        base=fit_reference(history,cutoff,choices).predict(keys)\n        models,blocks=fit_pair(history,cutoff,choices,weather)\n        preds=predictions_pair(keys,base,models,weather)\n        report[\'folds\'][label]={\'training_blocks\':blocks,\'variants\':{}}\n        for name,pred in preds.items():\n            m=score(keys,truth,pred);report[\'folds\'][label][\'variants\'][name]=m\n            print(f\'{label} {name}: WAPE-score={m["all"]["wape_score"]:.6f}\',flush=True)\n        if label==\'sep_oct\':\n            sep=[k for k in keys if k[1].month==9];october=[k for k in keys if k[1].month==10]\n            sept,_,_=select(sep,truth,preds,routes)\n            report[\'october_forward_diagnostic\']={\'choices_from_september\':sept,\n                \'metrics\':score(october,{k:truth[k] for k in october if k in truth},join(october,preds,sept))}\n            final_choice,calendar_choice,diag=select(keys,truth,preds,routes)\n            report[\'selection\']={\'by_route\':final_choice,\'calendar_control_by_route\':calendar_choice,\n                \'diagnostics\':diag,\'weather_accepted_routes\':[r for r,n in final_choice.items() if n.startswith(\'weather_\')]}\n            for name,ch in [(\'selected\',final_choice),(\'selected_calendar_control\',calendar_choice)]:\n                report[\'folds\'][label][\'variants\'][name]=score(keys,truth,join(keys,preds,ch))\n        dump(args.out/\'weather_report.json\',report)\n    reference=fit_reference(values,FINAL_CUTOFF,choices)\n    models,_=fit_pair(values,FINAL_CUTOFF,choices,weather)\n    model=WeatherEnsemble(reference,models,weather,final_choice)\n    base=reference.predict(sample);all_final=predictions_pair(sample,base,models,weather)\n    for name,ch in [(\'submission_competition.csv\',final_choice),(\'submission_calendar_control.csv\',calendar_choice)]:\n        report[name]=write_submission(args.sample,join(sample,all_final,ch),args.out/name)\n    # This must reproduce the protected 0.877 baseline byte-for-byte.\n    write_submission(args.sample,base,args.out/\'submission_reference.csv\')\n    if sha256(args.out/\'submission_reference.csv\')!=ref[\'submission_sha256\']:\n        raise RuntimeError(\'Frozen reference does not reproduce original CSV; stop and investigate\')\n    (args.out/\'model.local.pkl\').write_bytes(pickle.dumps(model,protocol=5))\n    report[\'outputs_sha256\']={n:sha256(args.out/n) for n in [\'submission_competition.csv\',\'submission_calendar_control.csv\',\'submission_reference.csv\',\'model.local.pkl\']}\n    final_predictions=join(sample,all_final,final_choice)\n    report[\'changed_rows_vs_reference\']=sum(final_predictions[k]!=base[k] for k in sample)\n    dump(args.out/\'weather_report.json\',report)\n    print(\'Weather accepted on routes:\',report[\'selection\'][\'weather_accepted_routes\'],flush=True)\n    print(\'Reference reproduced; 14640 rows exported. Competition score unknown.\',flush=True)\n    return report\n\nif __name__==\'__main__\':\n    from ml.forecast.weather_competition import main as run_main\n    run_main()\n', 'ml/forecast/reference_route_model.json': '{\n  "choices": {\n    "1": {\n      "candidate": "school_local",\n      "weight": 1.0,\n      "month_errors": [\n        41280,\n        55185\n      ]\n    },\n    "7": {\n      "candidate": "school_recent50",\n      "weight": 1.0,\n      "month_errors": [\n        83375,\n        78862\n      ]\n    },\n    "11": {\n      "candidate": "school_calendar",\n      "weight": 1.0,\n      "month_errors": [\n        90010,\n        92860\n      ]\n    },\n    "12": {\n      "candidate": "school_local",\n      "weight": 0.5,\n      "month_errors": [\n        87764,\n        84047\n      ]\n    },\n    "17": {\n      "candidate": "school_local",\n      "weight": 0.5,\n      "month_errors": [\n        86781,\n        120431\n      ]\n    },\n    "25": {\n      "candidate": "school_calendar",\n      "weight": 1.0,\n      "month_errors": [\n        29571,\n        49803\n      ]\n    },\n    "26": {\n      "candidate": "school_calendar",\n      "weight": 1.0,\n      "month_errors": [\n        43025,\n        59428\n      ]\n    },\n    "28": {\n      "candidate": "school_local",\n      "weight": 0.5,\n      "month_errors": [\n        42733,\n        39846\n      ]\n    },\n    "50": {\n      "candidate": "school_local",\n      "weight": 1.0,\n      "month_errors": [\n        133254,\n        136981\n      ]\n    }\n  },\n  "model_version": "ds1-route-63b89667a69c",\n  "validation_score": 0.8937443240536405,\n  "submission_sha256": "6efbbe7dc7dee64245b99573a7e92a1452a3816eb6e7cc4e2fc56e482663290f",\n  "selection_warning": "Choices were selected on Sep-Oct in prior research; all subsequent comparisons on that period remain exploratory."\n}\n', 'ml/forecast/test_weather_competition.py': '"""Small in-memory software tests; no persistent generated datasets."""\nimport unittest\nfrom datetime import date,timedelta\nfrom unittest.mock import patch\nimport numpy as np\nfrom ml.forecast.weather_competition import weather_rows,VARIABLES,corrected,select,join,causal_residual_training\n\nclass WeatherTests(unittest.TestCase):\n    def payload(self):\n        return {\'timezone\':\'Europe/Moscow\',\'utc_offset_seconds\':10800,\'hourly\':{\n            \'time\':[\'2025-01-01T00:00\',\'2025-01-01T01:00\'],\n            **{v:[1.,2.] for v in VARIABLES}}}\n\n    def test_missing_weather_is_not_silently_imputed(self):\n        p=self.payload();p[\'hourly\'][\'temperature_2m\'][1]=None\n        with self.assertRaisesRegex(ValueError,\'Missing\'):weather_rows(p)\n\n    def test_weather_time_alignment_and_rolling_values(self):\n        p=self.payload();r=weather_rows(p)\n        self.assertEqual(r[date(2025,1,1),1][-3:],[1.5,3.,0.])\n        p[\'hourly\'][\'time\'][1]=\'2025-01-01T02:00\'\n        with self.assertRaisesRegex(ValueError,\'contiguous\'):weather_rows(p)\n\n    def test_weather_must_beat_matched_control_in_both_months(self):\n        keys=[(\'1\',date(2025,m,1),12) for m in (9,10)]\n        truth=dict.fromkeys(keys,100)\n        preds={\'reference\':dict.fromkeys(keys,80)}\n        for w in (.25,.5,1.):\n            preds[f\'calendar_{w}\']=dict.fromkeys(keys,95)\n            preds[f\'weather_{w}\']={keys[0]:100,keys[1]:90}\n        chosen,control,_=select(keys,truth,preds,[\'1\'])\n        self.assertTrue(chosen[\'1\'].startswith(\'calendar\'))\n        preds[\'weather_0.5\']=dict.fromkeys(keys,99)\n        chosen,_,_=select(keys,truth,preds,[\'1\'])\n        self.assertEqual(chosen[\'1\'],\'weather_0.5\')\n\n    def test_corrections_are_bounded_and_route5_stays_zero(self):\n        keys=[(\'1\',date(2025,11,1),12),(\'5\',date(2025,11,1),12)]\n        base={keys[0]:100,keys[1]:0}\n        self.assertEqual(corrected(keys,base,[99,99],1),{keys[0]:135,keys[1]:0})\n        self.assertEqual(corrected(keys,base,[-99,-99],1),{keys[0]:65,keys[1]:0})\n\n    def test_residual_training_has_no_current_block_labels(self):\n        start=date(2025,1,1);end=date(2025,3,31)\n        history={(\'1\',start+timedelta(days=i),12):i+1 for i in range((end-start).days+1)}\n        seen=[]\n        def factory(prefix,cutoff,choices):\n            self.assertLessEqual(max(k[1] for k in prefix),cutoff)\n            class Reference:\n                def predict(_,keys):\n                    self.assertGreater(min(k[1] for k in keys),cutoff)\n                    seen.append((cutoff,min(k[1] for k in keys)))\n                    return dict.fromkeys(keys,10)\n            return Reference()\n        with patch(\'ml.forecast.weather_competition.fit_reference\',side_effect=factory):\n            keys,base,scale,target,blocks=causal_residual_training(history,end,{})\n        self.assertEqual(len(seen),2)\n        self.assertEqual(blocks[0][\'train_through\'],\'2025-01-31\')\n        self.assertEqual(keys[0][1],date(2025,2,1))\n\nif __name__==\'__main__\':unittest.main()\n', 'ml/forecast/weather_source.json': '{\n  "url": "https://archive-api.open-meteo.com/v1/archive?latitude=55.75&longitude=37.62&start_date=2025-01-01&end_date=2025-12-31&hourly=temperature_2m%2Crelative_humidity_2m%2Cprecipitation%2Crain%2Csnowfall%2Csnow_depth%2Cwind_speed_10m%2Ccloud_cover&timezone=Europe%2FMoscow&models=era5",\n  "source_mode": "competition",\n  "description": "Reanalysis, not available as a full future year at forecast cutoff",\n  "retrieved_at": "2026-09-26T19:25:02Z",\n  "attribution": "Weather data by Open-Meteo; ERA5 reanalysis by ECMWF/Copernicus."\n}\n', 'ml/forecast/network_regimes.py': '"""Route service states, sourced externally, with target-only causal profiles.\n\nCompetition mode deliberately allows retrospective transport notices. Dates in\nthese notices describe service, not a claim that recorded validations are zero.\n"""\nfrom __future__ import annotations\n\nimport json\nfrom collections import defaultdict\nfrom datetime import date\nfrom pathlib import Path\nfrom statistics import median\n\nfrom .core import days, effective_weekday, rounded\n\nNORMAL, JULY_SHORT, WEST_AUG, WEST_SEP, BOTH_ENDS, CANCELLED = range(6)\nSTATE_NAMES = (\'normal\', \'sokolniki_july\', \'novoslobodskaya_august\',\n               \'kalanchyovskaya\', \'kalanchyovskaya_and_bogorodskoe\', \'cancelled\')\nEVENTS = json.loads(Path(__file__).with_name(\'network_sources.json\').read_text())\n\n\ndef state(route, day, *, mode=\'competition\', as_of=None):\n    """A publication cutoff applies only to as-of mode; ends have own notices.\n\n    Repair weekends are literal Saturday/Sunday, independent of the demand\n    calendar. Nov 1/3/4 are uncertain and explicitly listed in source metadata.\n    """\n    if mode not in (\'competition\', \'as_of\') or (mode == \'as_of\' and as_of is None):\n        raise ValueError(\'Choose competition or as_of with a publication cutoff\')\n    route = str(route)\n    known = lambda stamp: mode == \'competition\' or date.fromisoformat(stamp) <= as_of\n    if route not in (\'7\', \'50\'):\n        return NORMAL\n    if route == \'7\' and known(\'2025-07-02\') and day >= date(2025, 7, 10):\n        if not (known(\'2025-08-13\') and day >= date(2025, 8, 13)):\n            return JULY_SHORT\n    if day.weekday() < 5:\n        return NORMAL\n    if known(\'2025-11-15\') and day >= date(2025, 11, 15):\n        return NORMAL\n    if known(\'2025-09-05\') and day >= date(2025, 9, 6):\n        if route == \'50\':\n            return CANCELLED\n        east = known(\'2025-09-19\') and day >= date(2025, 9, 20)\n        restored = known(\'2025-10-04\') and day >= date(2025, 10, 4)\n        return BOTH_ENDS if east and not restored else WEST_SEP\n    if route == \'7\' and known(\'2025-08-15\') and day >= date(2025, 8, 16):\n        return WEST_AUG\n    return NORMAL\n\n\nclass RegimeProfile:\n    """25% same-service hourly median + 75% frozen route reference.\n\n    Distinct shortened routes never borrow a July reduction coefficient.\n    Unseen or sparse states leave the reference untouched. Only announced\n    route-50 cancellations use a zero prior, or a learned residual-service\n    median once three same-weekday or four pooled weekend days are available.\n    Positive labels survive.\n    """\n    def __init__(self, weight=.25, mode=\'competition\', as_of=None):\n        if not 0 <= weight <= .5:\n            raise ValueError(\'Regime pooling weight must be in [0, .5]\')\n        self.weight, self.mode, self.as_of = weight, mode, as_of\n\n    def fit(self, history, start, cutoff):\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'History is empty or crosses cutoff\')\n        self.cutoff = cutoff\n        self.routes = {k[0] for k in history}\n        groups = defaultdict(list)\n        for r in (\'7\', \'50\'):\n            if r not in self.routes:\n                continue\n            for d in days(start, cutoff):\n                s = state(r, d, mode=self.mode, as_of=self.as_of)\n                # Weekdays remain the established route model. Holiday demand\n                # is not assumed to imply a repair closure.\n                if d.weekday() < 5:\n                    continue\n                for h in range(24):\n                    groups[r, s, d.weekday(), h].append(history.get((r, d, h), 0))\n                    groups[r, s, -1, h].append(history.get((r, d, h), 0))\n        self.support = {k: len(v) for k, v in groups.items()}\n        self.profiles = {k: float(median(v)) for k, v in groups.items()\n                         if len(v) >= (4 if k[2] == -1 else 3)}\n        return self\n\n    def lookup(self, route, service, weekday, hour):\n        return self.profiles.get((route, service, weekday, hour),\n                                 self.profiles.get((route, service, -1, hour)))\n\n    def apply(self, keys, base, *, active=(\'7\', \'50\'), zero_only=False):\n        if any(d <= self.cutoff for _, d, _ in keys):\n            raise ValueError(\'Predictions must follow training cutoff\')\n        out = {}\n        for k in keys:\n            r, d, h = k\n            b = base[k]\n            if r not in self.routes:\n                out[k] = 0\n                continue\n            s = state(r, d, mode=self.mode, as_of=self.as_of)\n            local = self.lookup(r, s, d.weekday(), h)\n            if r in active and s == CANCELLED:\n                # This is a prediction, never an overwrite of known truth.\n                out[k] = 0 if zero_only or local is None else rounded(local)\n            elif r in active and d.weekday() >= 5 and effective_weekday(d) == d.weekday() and local is not None:\n                # <=50% change in the profile before 25% pooling; protects\n                # restoration from the rejected DS2 history-exclusion rule.\n                local = min(1.5 * b, max(.5 * b, local))\n                out[k] = rounded((1 - self.weight) * b + self.weight * local)\n            else:\n                out[k] = b\n        return out\n\n    def coverage(self, keys):\n        counts = defaultdict(lambda: {\'rows\': 0, \'supported_rows\': 0})\n        for r, d, h in keys:\n            if r in (\'7\', \'50\'):\n                s = state(r, d, mode=self.mode, as_of=self.as_of)\n                item = counts[f\'{r}:{STATE_NAMES[s]}\']\n                item[\'rows\'] += 1\n                item[\'supported_rows\'] += self.lookup(r, s, d.weekday(), h) is not None\n        return dict(counts)\n', 'ml/forecast/network_competition.py': '"""Fixed-horizon competition comparison: service regimes x actual weather.\n\nOrganizer labels are verified by SHA256. No generated dataset can pass the\nentry point. New scores are produced only by running on those real inputs.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport calendar\nimport importlib.metadata\nimport json\nimport pickle\nfrom collections import defaultdict\nfrom datetime import date, datetime, timedelta, timezone\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .core import grid, load_labels, score, sha256, peak_hours\nfrom .network_regimes import RegimeProfile, state, STATE_NAMES, EVENTS\nfrom .route_specific import verify_inputs\nfrom .run import FIRST, FINAL_CUTOFF, FOLDS, dump\nfrom .weather_competition import (fit_reference, matrix, weather_rows,\n                                 corrected, WEIGHTS, select as select_weather)\nfrom ml.submit.adapter import read_sample, write_submission\n\n\ndef features(keys, base, weather, use_weather, network):\n    x = matrix(keys, base, weather, use_weather)\n    if network:\n        # Same service information is present in both matched weather arms.\n        x = np.column_stack((x, [state(r, d) for r, d, _ in keys]))\n    return x\n\n\nclass HistoryBank:\n    """Cache predictions for monthly blocks, fitted strictly before each block."""\n    def __init__(self, values, choices):\n        self.values, self.choices = values, choices\n        self.routes = sorted({k[0] for k in values}, key=int)\n        self.fits, self.blocks = {}, {}\n\n    def fitted(self, cutoff):\n        if cutoff not in self.fits:\n            history = {k: v for k, v in self.values.items() if FIRST <= k[1] <= cutoff}\n            self.fits[cutoff] = (fit_reference(history, cutoff, self.choices),\n                                RegimeProfile().fit(history, FIRST, cutoff))\n        return self.fits[cutoff]\n\n    def predict_bases(self, cutoff, keys):\n        ref, net = self.fitted(cutoff)\n        plain = ref.predict(keys)\n        return {\'plain\': plain, \'network\': net.apply(keys, plain)}\n\n    def training(self, cutoff):\n        keys, base = [], {\'plain\': {}, \'network\': {}}\n        audit = []\n        lo = date(2025, 2, 1)\n        while lo <= cutoff:\n            hi = min(cutoff, date(lo.year, lo.month, calendar.monthrange(lo.year, lo.month)[1]))\n            origin = lo - timedelta(days=1)\n            cache_key = (origin, hi)\n            if cache_key not in self.blocks:\n                batch = grid(self.routes, lo, hi)\n                self.blocks[cache_key] = (batch, self.predict_bases(origin, batch))\n            batch, predictions = self.blocks[cache_key]\n            keys.extend(batch)\n            for name in base:\n                base[name].update(predictions[name])\n            audit.append({\'train_through\': str(origin), \'predict_from\': str(lo), \'predict_to\': str(hi)})\n            lo = hi + timedelta(days=1)\n        if not keys:\n            raise ValueError(\'Need at least two months of history\')\n        # In this function the bank may contain future evaluation labels; only\n        # targets ending at cutoff are read. Each feature profile is earlier.\n        y = np.asarray([self.values.get(k, 0) for k in keys], dtype=float)\n        return keys, base, y, audit\n\n\ndef fit_models(bank, cutoff, weather):\n    import lightgbm as lgb\n    keys, bases, truth, audit = bank.training(cutoff)\n    models = {}\n    for group in (\'plain\', \'network\'):\n        base = bases[group]\n        scale = np.asarray([max(50, base[k]) for k in keys], dtype=float)\n        target = (truth - np.asarray([base[k] for k in keys])) / scale\n        models[group] = {}\n        for head, use_weather in ((\'calendar\', False), (\'weather\', True)):\n            x = features(keys, base, weather, use_weather, group == \'network\')\n            categorical = [0, 1, 2, 3, 4, 5] + ([x.shape[1]-1] if group == \'network\' else [])\n            reg = lgb.LGBMRegressor(objective=\'regression_l1\', n_estimators=220,\n                learning_rate=.025, num_leaves=12, min_child_samples=120,\n                reg_lambda=10., max_depth=5, n_jobs=2, verbosity=-1, random_state=2025)\n            reg.fit(x, target, sample_weight=scale, categorical_feature=categorical)\n            models[group][head] = reg\n    return models, audit\n\n\ndef predict_models(keys, bases, models, weather):\n    out = {}\n    for group, base in bases.items():\n        out[group] = {\'reference\': base}\n        for head, use_weather in ((\'calendar\', False), (\'weather\', True)):\n            raw = models[group][head].predict(features(keys, base, weather, use_weather, group == \'network\'))\n            for weight in WEIGHTS:\n                out[group][f\'{head}_{weight}\'] = corrected(keys, base, raw, weight)\n    return out\n\n\ndef metrics(keys, truth, predictions):\n    return score(keys, {k: truth[k] for k in keys if k in truth},\n                 {k: predictions[k] for k in keys})\n\n\ndef route_errors(keys, truth, prediction, route):\n    return sum(abs(prediction[k] - truth.get(k, 0)) for k in keys if k[0] == route)\n\n\ndef select_groups(keys, truth, preds, october_keys, october_truth, october_preds, routes):\n    """Finite, preregistered gate; no tuning of a state-specific effect size.\n\n    Route 50: >=0.5% gain in both long validation months. Route 7: the new\n    September endpoint is first learnable in the October refit, requiring\n    >=0.5% gain there and <=0.5% deterioration in each long validation month.\n    Both must also pass the October refit non-deterioration check.\n    """\n    chosen, audit = {}, {}\n    for r in routes:\n        monthly = [[k for k in keys if k[1].month == m] for m in (9, 10)]\n        errors = {g: [route_errors(x, truth, preds[g][\'reference\'], r) for x in monthly]\n                  for g in (\'plain\', \'network\')}\n        oct_errors = {g: route_errors(october_keys, october_truth, october_preds[g][\'reference\'], r)\n                      for g in (\'plain\', \'network\')}\n        wins = all(a <= .995*b for a, b in zip(errors[\'network\'], errors[\'plain\'])) and sum(errors[\'network\']) < sum(errors[\'plain\'])\n        learns_new_state = (r == \'7\' and oct_errors[\'network\'] < .995*oct_errors[\'plain\']\n                            and all(a <= 1.005*b for a, b in zip(errors[\'network\'], errors[\'plain\'])))\n        accepted = (r in (\'7\', \'50\') and (wins or learns_new_state)\n                    and oct_errors[\'network\'] <= 1.005*oct_errors[\'plain\'])\n        chosen[r] = \'network\' if accepted else \'plain\'\n        audit[r] = {\'long_month_errors\': errors, \'october_refit_errors\': oct_errors,\n                    \'passes_long_month_gate\': wins, \'passes_new_state_gate\': learns_new_state,\n                    \'selected\': chosen[r]}\n    return chosen, audit\n\n\ndef choose_heads(keys, truth, preds, groups, routes):\n    chosen, controls, audit = {}, {}, {}\n    for group in (\'plain\', \'network\'):\n        heads, no_weather, details = select_weather(keys, truth, preds[group], routes)\n        for r in routes:\n            if groups[r] == group:\n                chosen[r] = {\'group\': group, \'head\': heads[r]}\n                controls[r] = {\'group\': group, \'head\': no_weather[r]}\n                audit[r] = details[r]\n    return chosen, controls, audit\n\n\ndef assemble(keys, preds, choices):\n    return {k: preds[choices[k[0]][\'group\']][choices[k[0]][\'head\']][k]\n            if k[0] in choices else 0 for k in keys}\n\n\ndef slice_diagnostics(keys, truth, predictions, origin, history):\n    buckets = defaultdict(list)\n    peaks = peak_hours(history, sorted({k[0] for k in history}, key=int))\n    for k in keys:\n        r, d, h = k\n        lead = (d-origin).days\n        buckets[\'lead:\' + (\'01_14\' if lead <= 14 else \'15_30\' if lead <= 30 else \'31_61\')].append(k)\n        buckets[\'weekend\' if d.weekday() >= 5 else \'weekday\'].append(k)\n        if h in peaks.get(r, ()):\n            buckets[\'historical_peak_hours\'].append(k)\n        if r in (\'7\', \'50\'):\n            buckets[f\'route:{r}:state:{STATE_NAMES[state(r, d)]}\'].append(k)\n    out = {}\n    for name, subset in sorted(buckets.items()):\n        actual = sum(truth.get(k, 0) for k in subset)\n        error = sum(abs(predictions[k] - truth.get(k, 0)) for k in subset)\n        out[name] = {\'rows\': len(subset), \'actual_sum\': actual, \'absolute_error_sum\': error,\n                     \'wape_score\': max(0, 1-error/actual) if actual else None}\n    return out\n\n\nclass NetworkWeatherEnsemble:\n    def __init__(self, reference, regimes, models, weather, choices):\n        self.reference, self.regimes = reference, regimes\n        self.models, self.weather, self.choices = models, weather, choices\n\n    def predict(self, keys):\n        plain = self.reference.predict(keys)\n        bases = {\'plain\': plain, \'network\': self.regimes.apply(keys, plain)}\n        return assemble(keys, predict_models(keys, bases, self.models, self.weather), self.choices)\n\n\ndef main(argv=None):\n    p = argparse.ArgumentParser(description=__doc__)\n    for name in (\'train\', \'test\', \'sample\', \'weather\', \'reference\', \'out\'):\n        p.add_argument(\'--\'+name, type=Path, required=True)\n    args = p.parse_args(argv)\n    inputs = verify_inputs({n: getattr(args, n) for n in (\'train\', \'test\', \'sample\')})\n    values = load_labels([args.train, args.test])\n    if any(not FIRST <= k[1] <= FINAL_CUTOFF for k in values):\n        raise ValueError(\'Labels cross the allowed training interval\')\n    _, sample = read_sample(args.sample)\n    ref = json.loads(args.reference.read_text())\n    routes = sorted({k[0] for k in values}, key=int)\n    if set(routes) != set(ref[\'choices\']):\n        raise ValueError(\'Reference route set differs from labels\')\n    weather = weather_rows(json.loads(args.weather.read_text()))\n    if any((d, h) not in weather for _, d, h in grid([\'weather\'], FIRST, date(2025, 12, 31))):\n        raise ValueError(\'Weather must cover all 2025 hours\')\n    args.out.mkdir(parents=True, exist_ok=True)\n    report = {\'source_mode\': \'competition\', \'data_kind\': \'verified_organizer_labels\',\n        \'inputs\': inputs, \'reference\': ref, \'network_sources\': EVENTS,\n        \'research_audit\': json.loads(Path(__file__).with_name(\'research_audit.json\').read_text()),\n        \'weather_source\': json.loads(Path(__file__).with_name(\'weather_source.json\').read_text()),\n        \'weather_sha256\': sha256(args.weather), \'platform_score\': None,\n        \'selection_warning\': \'Exploratory: frozen route choices and new selection gates have seen Sep-Oct. October refit covers 31 days, not an independent 61-day test.\',\n        \'fitted_at\': datetime.now(timezone.utc).isoformat(),\n        \'libraries\': {n: importlib.metadata.version(n) for n in (\'numpy\', \'lightgbm\', \'scikit-learn\')},\n        \'code_sha256\': {f.name: sha256(f) for f in sorted(Path(__file__).parent.glob(\'*.py\'))},\n        \'folds\': {}, \'regime_weight\': .25, \'residual_correction_limit\': .35}\n    bank = HistoryBank(values, ref[\'choices\'])\n    experiments = list(FOLDS) + [(\'october_refit_31d\', date(2025, 9, 30), date(2025, 10, 1), date(2025, 10, 31))]\n    kept = {}\n    for label, cutoff, lo, hi in experiments:\n        print(f\'Fitting {label}; passenger labels through {cutoff}\', flush=True)\n        keys = grid(routes, lo, hi)\n        bases = bank.predict_bases(cutoff, keys)\n        models, training_audit = fit_models(bank, cutoff, weather)\n        preds = predict_models(keys, bases, models, weather)\n        variants = {f\'{g}/{n}\': metrics(keys, values, v) for g, heads in preds.items() for n, v in heads.items()}\n        net = bank.fitted(cutoff)[1]\n        for label_part, active in ((\'only_50\', (\'50\',)), (\'only_7\', (\'7\',))):\n            variants[label_part] = metrics(keys, values, net.apply(keys, bases[\'plain\'], active=active))\n        variants[\'50_zero_prior\'] = metrics(keys, values, net.apply(keys, bases[\'plain\'], active=(\'50\',), zero_only=True))\n        history = {k: y for k, y in values.items() if k[1] <= cutoff}\n        report[\'folds\'][label] = {\'cutoff\': str(cutoff), \'from\': str(lo), \'to\': str(hi),\n            \'days\': (hi-lo).days+1, \'training_blocks\': training_audit,\n            \'state_coverage\': net.coverage(keys), \'variants\': variants,\n            \'regime_slices\': slice_diagnostics(keys, values, bases[\'network\'], cutoff, history)}\n        for n in (\'plain/reference\', \'network/reference\', \'only_50\', \'only_7\'):\n            print(f\'{label} {n}: {variants[n]["all"]["wape_score"]:.6f}\', flush=True)\n        if label in (\'sep_oct\', \'october_refit_31d\'):\n            kept[label] = keys, preds\n        dump(args.out/\'network_weather_report.json\', report)\n\n    keys, preds = kept[\'sep_oct\']\n    oct_keys, oct_preds = kept[\'october_refit_31d\']\n    groups, group_audit = select_groups(keys, values, preds, oct_keys, values, oct_preds, routes)\n    choice, control, head_audit = choose_heads(keys, values, preds, groups, routes)\n    report[\'selection\'] = {\'by_route\': choice, \'without_weather_by_route\': control,\n        \'network_gate\': group_audit, \'weather_gate\': head_audit,\n        \'network_accepted_routes\': [r for r in routes if groups[r] == \'network\'],\n        \'weather_accepted_routes\': [r for r in routes if choice[r][\'head\'].startswith(\'weather_\')]}\n    for label, (ks, ps) in kept.items():\n        report[\'folds\'][label][\'variants\'][\'selected\'] = metrics(ks, values, assemble(ks, ps, choice))\n        report[\'folds\'][label][\'variants\'][\'selected_no_weather\'] = metrics(ks, values, assemble(ks, ps, control))\n    # October selection using September alone and a fixed Aug31 model remains\n    # a separate forward diagnostic, without the October refit\'s label access.\n    sep_keys = [k for k in keys if k[1].month == 9]\n    fixed_oct_keys = [k for k in keys if k[1].month == 10]\n    sep_groups = {r: (\'network\' if r in (\'7\', \'50\') and\n        route_errors(sep_keys, values, preds[\'network\'][\'reference\'], r) <\n        .995*route_errors(sep_keys, values, preds[\'plain\'][\'reference\'], r) else \'plain\') for r in routes}\n    sep_choice, _, _ = choose_heads(sep_keys, values, preds, sep_groups, routes)\n    report[\'october_forward_from_september\'] = {\'choices\': sep_choice,\n        \'metrics\': metrics(fixed_oct_keys, values, assemble(fixed_oct_keys, preds, sep_choice)),\n        \'warning\': \'Frozen reference was previously selected using Sep-Oct; only additional choices use September alone.\'}\n    # Historical restoration check corresponding to DS2 experiment 04.\n    restore_cut = date(2025, 8, 13)\n    restore_keys = grid(routes, date(2025, 8, 14), date(2025, 8, 31))\n    restore = bank.predict_bases(restore_cut, restore_keys)\n    report[\'august_restoration_check\'] = {n: metrics(restore_keys, values, v) for n, v in restore.items()}\n\n    reference, regimes = bank.fitted(FINAL_CUTOFF)\n    models, _ = fit_models(bank, FINAL_CUTOFF, weather)\n    bases = bank.predict_bases(FINAL_CUTOFF, sample)\n    predictions = predict_models(sample, bases, models, weather)\n    outputs = {\n        \'submission_network_weather.csv\': assemble(sample, predictions, choice),\n        \'submission_network_no_weather.csv\': assemble(sample, predictions, control),\n        \'submission_regimes_candidate.csv\': bases[\'network\'],\n        \'submission_reference.csv\': bases[\'plain\']}\n    # Keep a clearly labeled alternative for the uncertain repair schedule on\n    # the working Saturday and holiday weekdays, rather than silently asserting\n    # that the production calendar is a service timetable.\n    scenario = dict(outputs[\'submission_network_weather.csv\'])\n    mapping = {date(2025, 11, 1): date(2025, 11, 7),\n               date(2025, 11, 3): date(2025, 11, 2),\n               date(2025, 11, 4): date(2025, 11, 2)}\n    replacements = {k: (k[0], mapping[k[1]], k[2]) for k in sample if k[0] in (\'7\', \'50\') and k[1] in mapping}\n    # Deliberately simple demand proxies for this separate scenario: original\n    # Nov1 reference, and Nov2 restricted profile for Nov3/4. Weather corrections\n    # on those 144 rows are not retained. This is not the default submission.\n    for k, proxy in replacements.items():\n        if k[1] == date(2025, 11, 1):\n            scenario[k] = outputs[\'submission_reference.csv\'][k]\n        else:\n            scenario[k] = outputs[\'submission_regimes_candidate.csv\'][proxy]\n    outputs[\'submission_holiday_service_scenario.csv\'] = scenario\n    report[\'outputs\'] = {}\n    for name, prediction in outputs.items():\n        report[\'outputs\'][name] = write_submission(args.sample, prediction, args.out/name)\n    if sha256(args.out/\'submission_reference.csv\') != ref[\'submission_sha256\']:\n        raise RuntimeError(\'Frozen 0.877 reference did not reproduce byte-for-byte\')\n    ensemble = NetworkWeatherEnsemble(reference, regimes, models, weather, choice)\n    artifact = pickle.dumps(ensemble, protocol=5)\n    if pickle.loads(artifact).predict(sample) != outputs[\'submission_network_weather.csv\']:\n        raise RuntimeError(\'Saved model prediction mismatch\')\n    (args.out/\'model.local.pkl\').write_bytes(artifact)\n    report[\'outputs_sha256\'] = {n: sha256(args.out/n) for n in (*outputs, \'model.local.pkl\')}\n    report[\'changed_rows_vs_reference\'] = {n: sum(v[k] != bases[\'plain\'][k] for k in sample) for n, v in outputs.items()}\n    report[\'final_state_coverage\'] = regimes.coverage(sample)\n    report[\'restoration_warning\'] = \'Nov15 service restoration is confirmed externally. Its hourly demand profile is pooled with earlier normal-service history; no November demand truth is available.\'\n    report[\'holiday_service_scenario\'] = \'Alternative only: Nov1 normal service, Nov3/4 Sunday restricted service for 7/50. No claim that this unpublished repair schedule is correct.\'\n    dump(args.out/\'network_weather_report.json\', report)\n    print(\'Accepted service regimes:\', report[\'selection\'][\'network_accepted_routes\'], flush=True)\n    print(\'Accepted weather:\', report[\'selection\'][\'weather_accepted_routes\'], flush=True)\n    print(\'Saved 14640-row submissions; route 5 retained with zeros. Platform score unknown.\', flush=True)\n    return report\n\n\nif __name__ == \'__main__\':\n    from ml.forecast.network_competition import main as run_main\n    run_main()\n', 'ml/forecast/test_network_competition.py': '"""In-memory correctness checks, not a demand-quality benchmark."""\nimport json\nimport pickle\nimport unittest\nfrom datetime import date, timedelta\nfrom unittest.mock import patch\n\nimport numpy as np\n\nfrom ml.forecast.core import grid\nfrom ml.forecast.network_regimes import (state, NORMAL, JULY_SHORT, WEST_AUG, WEST_SEP,\n                              BOTH_ENDS, CANCELLED, RegimeProfile)\nfrom ml.forecast.network_competition import HistoryBank, select_groups, assemble, features\n\n\nclass NetworkTests(unittest.TestCase):\n    def test_distinct_endpoints_and_restoration_boundaries(self):\n        for stamp, expected in [(\'2025-07-12\', JULY_SHORT), (\'2025-08-13\', NORMAL),\n                (\'2025-08-16\', WEST_AUG), (\'2025-09-06\', WEST_SEP),\n                (\'2025-09-20\', BOTH_ENDS), (\'2025-10-04\', WEST_SEP),\n                (\'2025-11-09\', WEST_SEP), (\'2025-11-15\', NORMAL)]:\n            self.assertEqual(state(\'7\', date.fromisoformat(stamp)), expected, stamp)\n        self.assertEqual(state(\'50\', date(2025, 9, 6)), CANCELLED)\n        self.assertEqual(state(\'50\', date(2025, 11, 15)), NORMAL)\n\n    def test_demand_holiday_does_not_imply_repair_closure(self):\n        self.assertEqual(state(\'50\', date(2025, 11, 1)), CANCELLED)  # literal Saturday\n        self.assertEqual(state(\'50\', date(2025, 11, 3)), NORMAL)  # demand holiday\n        self.assertEqual(state(\'50\', date(2025, 9, 8)), NORMAL)\n        self.assertEqual(state(\'17\', date(2025, 9, 6)), NORMAL)\n\n    def test_asof_uses_separate_publications_for_start_and_end(self):\n        d = date(2025, 11, 22)\n        self.assertEqual(state(\'50\', d, mode=\'as_of\', as_of=date(2025, 8, 31)), NORMAL)\n        self.assertEqual(state(\'50\', d, mode=\'as_of\', as_of=date(2025, 10, 31)), CANCELLED)\n        self.assertEqual(state(\'50\', d), NORMAL)\n        self.assertEqual(state(\'7\', date(2025, 10, 11), mode=\'as_of\', as_of=date(2025, 9, 30)), BOTH_ENDS)\n        with self.assertRaises(ValueError):\n            state(\'7\', d, mode=\'as_of\')\n\n    def test_unseen_state_does_not_transfer_july_factor(self):\n        keys = grid([\'7\', \'50\', \'1\'], date(2025, 1, 1), date(2025, 8, 31))\n        history = {k: 10 if state(k[0], k[1]) == JULY_SHORT else 100 for k in keys}\n        model = RegimeProfile().fit(history, date(2025, 1, 1), date(2025, 8, 31))\n        forecast = grid([\'7\', \'50\', \'1\', \'5\'], date(2025, 9, 6), date(2025, 9, 6))\n        base = dict.fromkeys(forecast, 100)\n        pred = model.apply(forecast, base)\n        self.assertEqual(pred[\'7\', date(2025, 9, 6), 12], 100)\n        self.assertEqual(pred[\'50\', date(2025, 9, 6), 12], 0)\n        self.assertEqual(pred[\'1\', date(2025, 9, 6), 12], 100)\n        self.assertEqual(pred[\'5\', date(2025, 9, 6), 12], 0)\n\n    def test_cancellation_keeps_observed_truth_and_learns_residual(self):\n        history = {k: 12 for k in grid([\'50\'], date(2025, 9, 6), date(2025, 10, 31))}\n        original = dict(history)\n        model = RegimeProfile().fit(history, date(2025, 9, 6), date(2025, 10, 31))\n        keys = [(\'50\', date(2025, 11, 2), 12)]\n        self.assertEqual(model.apply(keys, dict.fromkeys(keys, 100))[keys[0]], 12)\n        self.assertEqual(history, original)\n\n    def test_september_state_pools_four_weekend_days_for_october(self):\n        keys = grid([\'7\'], date(2025, 9, 1), date(2025, 9, 30))\n        history = {k: 40 if state(\'7\', k[1]) == WEST_SEP else 100 for k in keys}\n        model = RegimeProfile().fit(history, date(2025, 9, 1), date(2025, 9, 30))\n        future = [(\'7\', date(2025, 10, 4), 12)]\n        self.assertEqual(model.lookup(\'7\', WEST_SEP, 5, 12), 40)\n        self.assertEqual(model.apply(future, dict.fromkeys(future, 100))[future[0]], 88)\n\n    def test_restoration_removes_zero_rule_and_pooling_is_bounded(self):\n        keys = grid([\'50\'], date(2025, 1, 1), date(2025, 10, 31))\n        history = {k: 0 if state(k[0], k[1]) == CANCELLED else 1000 for k in keys}\n        model = RegimeProfile().fit(history, date(2025, 1, 1), date(2025, 10, 31))\n        future = [(\'50\', date(2025, 11, 9), 12), (\'50\', date(2025, 11, 15), 12)]\n        pred = model.apply(future, dict.fromkeys(future, 100))\n        self.assertEqual(pred[future[0]], 0)\n        self.assertEqual(pred[future[1]], 112)  # 25% of bounded 150 profile; half-even\n        self.assertEqual(pickle.loads(pickle.dumps(model)).apply(future, dict.fromkeys(future, 100)), pred)\n\n    def test_state_training_cannot_cross_cutoff(self):\n        with self.assertRaises(ValueError):\n            RegimeProfile().fit({(\'7\', date(2025, 9, 1), 12): 1}, date(2025, 1, 1), date(2025, 8, 31))\n\n    def test_training_cache_never_reads_future_labels_into_features_or_targets(self):\n        values = {(\'1\', date(2025, 1, 1)+timedelta(days=i), 12): i+1 for i in range(120)}\n        seen = []\n        def factory(prefix, cutoff, choices):\n            self.assertLessEqual(max(k[1] for k in prefix), cutoff)\n            class Model:\n                def predict(_, ks):\n                    self.assertGreater(min(k[1] for k in ks), cutoff)\n                    seen.append(cutoff)\n                    return dict.fromkeys(ks, sum(prefix.values()))\n            return Model()\n        with patch(\'ml.forecast.network_competition.fit_reference\', side_effect=factory):\n            a = HistoryBank(values, {}).training(date(2025, 3, 31))\n            perturbed = {k: (999999 if k[1] > date(2025, 3, 31) else v) for k, v in values.items()}\n            b = HistoryBank(perturbed, {}).training(date(2025, 3, 31))\n        self.assertEqual(a[0:2], b[0:2])\n        np.testing.assert_array_equal(a[2], b[2])\n        self.assertEqual(set(seen), {date(2025, 1, 31), date(2025, 2, 28)})\n\n    def test_bad_long_month_rejects_route50_even_with_good_october_refit(self):\n        ks = [(\'50\', date(2025, m, 6), 12) for m in (9, 10)]\n        truth = dict.fromkeys(ks, 100)\n        preds = {\'plain\': {\'reference\': dict.fromkeys(ks, 80)},\n                 \'network\': {\'reference\': {ks[0]: 100, ks[1]: 70}}}\n        recent = {\'plain\': {\'reference\': dict.fromkeys(ks, 80)},\n                  \'network\': {\'reference\': dict.fromkeys(ks, 100)}}\n        result, _ = select_groups(ks, truth, preds, ks, truth, recent, [\'50\'])\n        self.assertEqual(result[\'50\'], \'plain\')\n\n    def test_new_route7_state_requires_october_evidence(self):\n        ks = [(\'7\', date(2025, m, 6), 12) for m in (9, 10)]\n        truth = dict.fromkeys(ks, 100)\n        preds = {g: {\'reference\': dict.fromkeys(ks, 80)} for g in (\'plain\', \'network\')}\n        recent = {\'plain\': {\'reference\': dict.fromkeys(ks, 80)},\n                  \'network\': {\'reference\': dict.fromkeys(ks, 98)}}\n        result, _ = select_groups(ks, truth, preds, ks, truth, recent, [\'7\'])\n        self.assertEqual(result[\'7\'], \'network\')\n\n    def test_weather_pair_has_identical_service_feature(self):\n        ks = [(\'50\', date(2025, 9, 6), 12)]\n        weather = {(ks[0][1], 12): [0.]*10}\n        base = dict.fromkeys(ks, 0)\n        a = features(ks, base, weather, False, True)\n        b = features(ks, base, weather, True, True)\n        self.assertEqual(a[0, -1], CANCELLED)\n        self.assertEqual(b[0, -1], CANCELLED)\n        np.testing.assert_array_equal(a[:, :9], b[:, :9])\n\n    def test_unknown_route_preserved_as_zero_when_assembling(self):\n        ks = [(\'5\', date(2025, 11, 1), 12)]\n        self.assertEqual(assemble(ks, {}, {}), {ks[0]: 0})\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'ml/forecast/network_sources.json': '{\n  "source_mode": "competition",\n  "retrieved_at": "2026-09-27",\n  "research_branch": "ds2/research-protocol",\n  "research_commit": "b593ebbb5351bcc31881a99431d820ad97b5d140",\n  "authorization": "Organizer Q&A permits external facts published after 2025-10-31, including actual weather and transport news. Future passenger targets are never permitted.",\n  "events": [\n    {"routes": [7], "published_at": "2025-07-02", "start": "2025-07-10", "end_exclusive": "2025-08-13", "end_published_at": "2025-08-13", "state": "sokolniki_july", "days": "all", "source": "https://transport.mos.ru/mostrans/all_news/125169", "end_source": "https://transport.mos.ru/mostrans/all_news/125800"},\n    {"routes": [7], "published_at": "2025-08-15", "start": "2025-08-16", "end_exclusive": "2025-09-06", "end_published_at": "2025-09-05", "state": "novoslobodskaya_august", "days": "Saturday/Sunday", "source": "https://transport.mos.ru/mostrans/all_news/125846"},\n    {"routes": [7, 50], "published_at": "2025-09-05", "start": "2025-09-06", "end_exclusive": "2025-11-15", "end_published_at": "2025-11-15", "state_by_route": {"7": "kalanchyovskaya", "50": "cancelled"}, "days": "Saturday/Sunday", "source": "https://mosmetro.ru/news/details/7732", "readable_primary_mirror": "https://mosmetro.teletribe.ru/news/details/7732", "end_source": "https://t.me/DtOperativno/23565", "note": "DS2 registry omitted the simultaneous route-7 change in this route-50 notice; included here. Recorded positive route-50 labels are retained."},\n    {"routes": [7], "published_at": "2025-09-19", "start": "2025-09-20", "end_exclusive": "2025-10-04", "end_published_at": "2025-10-04", "state": "kalanchyovskaya_and_bogorodskoe", "days": "Saturday/Sunday", "source": "https://transport.mos.ru/mostrans/all_news/126405", "end_source": "https://mosmetro.ru/news/details/7973", "readable_end_source": "https://mosmetro.teletribe.ru/news/details/7973"}\n  ],\n  "limitations": [\n    "This is a partial daytime service registry, not a complete historical timetable.",\n    "Repair weekends are interpreted literally as Saturday/Sunday. Nov 1 is a working Saturday: whether repair closures followed the changed work calendar is unconfirmed. Nov 3/4 closures are not inferred from the public-holiday demand calendar. A separate holiday-service scenario CSV isolates this uncertainty.",\n    "Late-night changes reported Sep 12, Oct 28, Nov 5 and Dec 13 have no fully verified repeat schedule and hour-by-hour capacity. No invented zero-service mask is applied to them.",\n    "November restoration is externally confirmed, but no November labels exist locally; its passenger-demand effect is unvalidated.",\n    "July-shortening and September-east-detour magnitudes are not assumed identical. An unseen state falls back to the reference.",\n    "The 31-day October refit diagnostic can assess a state observed in September; it is not an independent 61-day test. The long Sep-Oct gate remains reported separately."\n  ]\n}\n', 'ml/forecast/research_audit.json': '{\n  "branch": "ds2/research-protocol",\n  "commit": "b593ebbb5351bcc31881a99431d820ad97b5d140",\n  "scope": "docs/research recursively, complete tracked files",\n  "review": "All Markdown reports and Python scripts inspected; all saved JSON reports parsed and metric identities checked. Experiments were not rerun without organizer labels.",\n  "file_count": 56,\n  "counts": {\n    ".md": 19,\n    ".py": 19,\n    ".json": 18\n  },\n  "metric_identity_checks": 1063,\n  "files": [\n    {\n      "path": "CASE_STUDIES_2025.md",\n      "sha256": "9a4984e5d807f4e63b16e20f03c2b5024b1a55af7ecd4588a40b9226ddb3d166",\n      "bytes": 7616\n    },\n    {\n      "path": "EXPERIMENT_02_HISTORY.md",\n      "sha256": "7740029d69ccb1ae6140aba1aad85af7a95146ab37125da37f584713c920ba49",\n      "bytes": 5964\n    },\n    {\n      "path": "EXPERIMENT_03_CALENDAR.md",\n      "sha256": "211abd3b6fb18c61d68cf396e1d51fd0d26bb61682cbe0adbddb54cdd73096c9",\n      "bytes": 4749\n    },\n    {\n      "path": "EXPERIMENT_04_ROUTE7_RESTORATION.md",\n      "sha256": "67b3b013deb111ed9091b5dcb4cfe804c2f384c105b493de466088265817908c",\n      "bytes": 5865\n    },\n    {\n      "path": "EXPERIMENT_05_ROUTE50_SHUTDOWN.md",\n      "sha256": "daf0259ba7a538a05762d53a7365f853b5179b8e8641013d1a9a8f36a8e014ae",\n      "bytes": 6491\n    },\n    {\n      "path": "EXPERIMENT_06_DS1_REAL_LABELS.md",\n      "sha256": "00ae445c9ce6a928baef74cf8484c330032d5527daf87e6394842b39607dce14",\n      "bytes": 5505\n    },\n    {\n      "path": "EXPERIMENT_07_ADAPTIVE_PROFILES.md",\n      "sha256": "309bb442b43676bbc67986ceff39f4e3422b64462e263b41defff3ac5b3cc728",\n      "bytes": 6083\n    },\n    {\n      "path": "EXPERIMENT_08_NETWORK_AND_SOURCE_AUDIT.md",\n      "sha256": "e7bc6d4096534a4bdee2a3635665f2329de618f1779d18d1941ec4a56c66c7bf",\n      "bytes": 10484\n    },\n    {\n      "path": "EXPERIMENT_09_SCHEDULE_AND_GAPS.md",\n      "sha256": "a976453ebb2b3ab9df9f592cab345faae37158d21fc7cd829facde9460283161",\n      "bytes": 9534\n    },\n    {\n      "path": "EXPERIMENT_10_SCHOOL_TERM.md",\n      "sha256": "274c2fa9ffcb55fc0f3dc1493bd38d4b1372b6d70abb44dc525bef8840470eb4",\n      "bytes": 5512\n    },\n    {\n      "path": "EXPERIMENT_11_ANNUAL_SEASONALITY.md",\n      "sha256": "72ba8ceb60b0824e79bda4248c342fb54e44fcdddb94ea5705449fbf7e16e2d7",\n      "bytes": 3243\n    },\n    {\n      "path": "EXPERIMENT_12_EXTERNAL_AND_YEAR.md",\n      "sha256": "6d23415deed545c7503a71cf5f9db0462f5ef215cd2c199260a9b23d411424a6",\n      "bytes": 8568\n    },\n    {\n      "path": "EXPERIMENT_13_AUGUST_REGIME.md",\n      "sha256": "6c0b1e0fbcfea3cb72c4475d1d4d5c8bf78b7d2630c6dfd5008db0c3756ba1c6",\n      "bytes": 5239\n    },\n    {\n      "path": "EXPERIMENT_14_NEAR_TERM.md",\n      "sha256": "f2024714ca248299d4daa3d933c722e012b77de27a570c5d41aeb367508c5c1e",\n      "bytes": 5498\n    },\n    {\n      "path": "EXPERIMENT_15_SCHOOL_POI_ASOF.md",\n      "sha256": "e54058d79116ef88f98cf5c5e769315e15dac1209876f60b86b6fe92166a954b",\n      "bytes": 6852\n    },\n    {\n      "path": "FACTOR_MATRIX.md",\n      "sha256": "888c73c456a94a51cb678dfca0031ebadd465783295b469feff22cffa0a9a1c4",\n      "bytes": 18276\n    },\n    {\n      "path": "MODEL_CRITERIA_2026-09-26.md",\n      "sha256": "737ccbdbbe4d434e99905688bc5cd282ef0ad83656f584238ba515e1603c34bf",\n      "bytes": 9542\n    },\n    {\n      "path": "README.md",\n      "sha256": "a8cd5e3b21a814ec8ad027866ff3c3218e552837a2f37e0deb427246c9fe75d4",\n      "bytes": 7066\n    },\n    {\n      "path": "RESULTS_2026-09-26.md",\n      "sha256": "f8dcd389b856420bcec8ad2ac4cdef60913245660c457ffdba718fc94debc85d",\n      "bytes": 8576\n    },\n    {\n      "path": "audit_missing_hours_raw.py",\n      "sha256": "4f8821f3d2ec9dd8ef26de8505bb81e0bfefcebf59124dec8c8e5db1635cfb45",\n      "bytes": 4478,\n      "functions": [\n        "days",\n        "main"\n      ]\n    },\n    {\n      "path": "audit_route50_sep13.py",\n      "sha256": "65e53f7b197309e06e4a62ef8e6a2e93999e77d8d5d344a43bfb118b5bb2f521",\n      "bytes": 3646,\n      "functions": [\n        "main"\n      ]\n    },\n    {\n      "path": "backtest_baseline.py",\n      "sha256": "3f28f66e3e2e095058838d0a94b3b38e7a8aea04e30c43fb99ecbb907a607f65",\n      "bytes": 5320,\n      "functions": [\n        "load",\n        "dates",\n        "fit",\n        "evaluate",\n        "main"\n      ]\n    },\n    {\n      "path": "diagnose_routes.py",\n      "sha256": "a2c24e59a3442bf3f39028926f0f313b92e38897f894794a5aaaa7f8f6f0ce80",\n      "bytes": 3848,\n      "functions": [\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_14_near_term.py",\n      "sha256": "64063003ab70a52d6bd4d869f33821bd9ed7ed593a2e462a544ef2f404bc88e3",\n      "bytes": 5670,\n      "functions": [\n        "load",\n        "windows",\n        "measure",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_15_school_poi.py",\n      "sha256": "6367e8a2bd14608239f1d5f6299337a8a7efd8766f908ae39d114f79efac35b8",\n      "bytes": 5564,\n      "functions": [\n        "days",\n        "read_labels",\n        "profile_score",\n        "audited_stops",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_adaptive_profiles.py",\n      "sha256": "bea122b940756225faddab502da31d271b11c450134eed4c168fddcd91fed3f7",\n      "bytes": 6239,\n      "functions": [\n        "recent_means",\n        "predict",\n        "assess",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_annual_fourier.py",\n      "sha256": "dfc97e68ff989d510dd79cdbd42cf7d8c8119e63f94071e0baba6292f7f59d2d",\n      "bytes": 4192,\n      "functions": [\n        "vector",\n        "run_fold",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_august_regime.py",\n      "sha256": "beb45b83bc205430c431b95a269cfc700e17e7e8e8e3a6d32fff7c0fdb5e6523",\n      "bytes": 4057,\n      "functions": [\n        "run",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_calendar.py",\n      "sha256": "1a1dbb0747cbb789d3ef7f226ca73dc74073ef64147bbb6722a59f94afe543a1",\n      "bytes": 4858,\n      "functions": [\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_ds1_candidates.py",\n      "sha256": "e7a199335e7e8b085037ce16510744dd1c3ebdd83d5a07b31df8fe39cc41ed4f",\n      "bytes": 4347,\n      "functions": [\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_history_window.py",\n      "sha256": "fe83d4c96179fe0ecbd2605092a4e3a7ea946c685ef83651b2cfef948a35fb0b",\n      "bytes": 3105,\n      "functions": [\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_network_sept19.py",\n      "sha256": "c777658e3881c929eb0f8fca860133e135b3747c8676ed1ee8f0c9d19a952ac5",\n      "bytes": 5157,\n      "functions": [\n        "predict",\n        "route7_transfer_factor",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_route17_restoration.py",\n      "sha256": "abe7b995edbcd2f6dbbb3407d20aa93c65d0de8c9921372f8579d0dcd763cae2",\n      "bytes": 4223,\n      "functions": [\n        "revised_route17_profile",\n        "metrics",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_route50_shutdown.py",\n      "sha256": "8ac9891d653652bb48b1e24ec8a03badc36072b54e8b54be62b3aec3f1878111",\n      "bytes": 4174,\n      "functions": [\n        "compare",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_route7_restoration.py",\n      "sha256": "7edd3ece4cc68f917722d4d7316acb5fa82ab43f13f96c35beedf793c29b5c4d",\n      "bytes": 4823,\n      "functions": [\n        "fit_with_known_status",\n        "score",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_school_term_profiles.py",\n      "sha256": "1a25b05c95b5c608d0a436b5257628e1a2fa0bd872764ad70c75c37974fa5e5e",\n      "bytes": 3821,\n      "functions": [\n        "score",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_year_horizon.py",\n      "sha256": "97831b55945f01de9984d3fbd869f60a1ff09a0387df5a5e1287ce2319741338",\n      "bytes": 3013,\n      "functions": [\n        "peak_metrics",\n        "main"\n      ]\n    },\n    {\n      "path": "profile_data.py",\n      "sha256": "30db2ba842ee830a8212a31edf8db048b049f7dbe1ebe0849e6a484c96d8d0e4",\n      "bytes": 10200,\n      "functions": [\n        "open_text",\n        "parse_timestamp",\n        "parse_route",\n        "write_csv",\n        "load_labels",\n        "inspect_submission",\n        "days",\n        "audit_raw",\n        "main"\n      ]\n    },\n    {\n      "path": "results/adaptive_profiles.json",\n      "sha256": "b87dae1e03381c8556911f08fbc926c7836fe71f5f406323e411b6e7cc1b5bfa",\n      "bytes": 44543,\n      "top_level_fields": [\n        "hypothesis",\n        "calendar_source",\n        "candidate_grid",\n        "missing_label_rule",\n        "folds",\n        "selection"\n      ]\n    },\n    {\n      "path": "results/annual_fourier.json",\n      "sha256": "d6cd86d8f4e911121539c21c93592299099767e77c874b91d036d75a3c0d1bda",\n      "bytes": 3458,\n      "top_level_fields": [\n        "calendar_features",\n        "network_scale_routes",\n        "excluded_from_scale",\n        "rules",\n        "folds"\n      ]\n    },\n    {\n      "path": "results/august_regime.json",\n      "sha256": "79c0009fcd49255c4d69626c13b9438543bc83ce4d47e6d66248579eaf15d7e7",\n      "bytes": 5314,\n      "top_level_fields": [\n        "cutoff",\n        "validation",\n        "status",\n        "source",\n        "metrics"\n      ]\n    },\n    {\n      "path": "results/backtest.json",\n      "sha256": "f711379ea7e8ebb23b94d8b259a833c4f1cd7c40e6b00178d2a47f1003994589",\n      "bytes": 7279,\n      "top_level_fields": [\n        "method",\n        "missing_label_rule",\n        "routes",\n        "folds"\n      ]\n    },\n    {\n      "path": "results/calendar.json",\n      "sha256": "d84c45e9612a3ed23f49f52cb306f55cdd5409f2ac5940fb411a1d25f8ed509e",\n      "bytes": 4178,\n      "top_level_fields": [\n        "source_url",\n        "published_at",\n        "hypothesis",\n        "new_feature",\n        "routes",\n        "folds"\n      ]\n    },\n    {\n      "path": "results/ds1_real_candidates.json",\n      "sha256": "8bdfc73f7a63f1b70563b6ad84bfc932116897bc7c9ceee6184875af7b4dc6d1",\n      "bytes": 90696,\n      "top_level_fields": [\n        "source_branch",\n        "source_commit",\n        "ds1_code_sha256",\n        "input_sha256",\n        "metric",\n        "policy",\n        "folds",\n        "selection"\n      ]\n    },\n    {\n      "path": "results/experiment_15_school_poi_asof.json",\n      "sha256": "0becf8b5167656b2a82f9d074e98ea2f81a1a283291f2fdb1f2ac23b7f6731fd",\n      "bytes": 1083,\n      "top_level_fields": [\n        "cutoff",\n        "holdout",\n        "hour_grid",\n        "baseline",\n        "exclude_summer_weekdays",\n        "delta_score",\n        "selection_note",\n        "stops"\n      ]\n    },\n    {\n      "path": "results/history_window.json",\n      "sha256": "595313903aabbc72610249572c09d4b88ef331a99792faf063e8821001bcdf1c",\n      "bytes": 15558,\n      "top_level_fields": [\n        "hypothesis",\n        "changed_parameter",\n        "leakage_rule",\n        "routes",\n        "folds"\n      ]\n    },\n    {\n      "path": "results/missing_hours_raw.json",\n      "sha256": "f41b898af29f924914683b71ed20049f419ca1844fec63e9f429749d6bf87554",\n      "bytes": 13378,\n      "top_level_fields": [\n        "raw_rows_read",\n        "scope",\n        "candidate_hours_not_in_labels",\n        "candidate_hours_with_any_raw_transaction",\n        "candidate_hours_with_success",\n        "oracle_125_hour_error_removed_from_aug31_median",\n        "oracle_score_gain_if_future_missing_keys_known",\n        "raw_status_totals",\n        "raw_transaction_types",\n        "missing_hours",\n        "limitations"\n      ]\n    },\n    {\n      "path": "results/network_events_2025.json",\n      "sha256": "8659d4aee1b2b4b7e17ab91b0099f885f048ce5b5bd98d5bdf573d3b50bcbd84",\n      "bytes": 3742,\n      "top_level_fields": [\n        "scope",\n        "time_rule",\n        "routes_in_labels",\n        "routes_with_events_in_this_partial_registry",\n        "events"\n      ]\n    },\n    {\n      "path": "results/network_sept19.json",\n      "sha256": "fc70fd54fc6eed72cba400ebb571fab6e95dfe61a20cd203ca35e723b420aef2",\n      "bytes": 11711,\n      "top_level_fields": [\n        "cutoff",\n        "validation",\n        "event_dates",\n        "news",\n        "route7_scenario_transfer",\n        "training_rule",\n        "predictions",\n        "metrics"\n      ]\n    },\n    {\n      "path": "results/route17_restoration.json",\n      "sha256": "ed58b4ad0ee8c338c23b4a6ca37947274b18d78cbe09dfd07412f1cf7b2ee00c",\n      "bytes": 17182,\n      "top_level_fields": [\n        "sources",\n        "restricted_weekends_excluded_from_train",\n        "effect_isolated_to_route",\n        "folds"\n      ]\n    },\n    {\n      "path": "results/route50_sep13_raw_audit.json",\n      "sha256": "493cdf647c8dd8b50b78a55c953fe87e22a18d91acbf0112ace528a8b7c4a483",\n      "bytes": 4348,\n      "top_level_fields": [\n        "source",\n        "rows_scanned",\n        "route_50_invalid_event_time",\n        "sep13_status_counts",\n        "sep13_status_transaction_type_counts",\n        "sep13_hourly",\n        "weekend_daily"\n      ]\n    },\n    {\n      "path": "results/route50_shutdown.json",\n      "sha256": "12d4dfa2ee886b2e04fca1424442a2a0e83959b147e3d370405d32ed7e50a6fe",\n      "bytes": 9622,\n      "top_level_fields": [\n        "source",\n        "published_at",\n        "cutoff",\n        "validation",\n        "missing_label_rule",\n        "changed_rows_per_statistic",\n        "variants",\n        "affected_day_facts"\n      ]\n    },\n    {\n      "path": "results/route7_restoration.json",\n      "sha256": "d71ee09e86ed79bb323deac50b1feb7644ca3f3906be40eb32f0652bd4339e47",\n      "bytes": 8203,\n      "top_level_fields": [\n        "source_start",\n        "start_notice_published",\n        "source_restored",\n        "restoration_notice_published",\n        "cutoff",\n        "excluded_route_7_training_dates",\n        "validation",\n        "definition",\n        "variants"\n      ]\n    },\n    {\n      "path": "results/route_diagnostics.json",\n      "sha256": "a904ee4732a186530b9e1d47dad50867141d4e6b00a4083c43f26bce532bace3",\n      "bytes": 3603,\n      "top_level_fields": [\n        "method",\n        "candidate_service_hours",\n        "routes_in_labels",\n        "route_7_windows",\n        "missing_active_hours_count",\n        "missing_active_hours_by_route",\n        "missing_active_hours_by_date"\n      ]\n    },\n    {\n      "path": "results/school_term_profiles.json",\n      "sha256": "6a0d0b7d7c860b14b1f876fa7f0bee1fa0ef328d526bbee32f339b843903570d",\n      "bytes": 8347,\n      "top_level_fields": [\n        "train_cutoff",\n        "validation",\n        "rule",\n        "selection_warning",\n        "baseline_median",\n        "variants"\n      ]\n    },\n    {\n      "path": "results/year_horizon.json",\n      "sha256": "17d534587150c0b44d37c00eb85e4a7638c7e1c26a9eb7c1cf7e1bef0c1791dd",\n      "bytes": 13960,\n      "top_level_fields": [\n        "metric",\n        "model",\n        "external_sources",\n        "full_year_validated",\n        "folds"\n      ]\n    }\n  ]\n}\n'}
SOURCE_HASHES = {'ml/forecast/core.py': '819cfdc88d5b6b96bf80fb0e019eeef0678628b617d2dd077409d26265369b5f', 'ml/forecast/models.py': 'd02f6120641005057c411d120894617027ff8b6a10a0afba37a8e5612b5e2009', 'ml/forecast/route_specific.py': '3f9a0f92eb4bdccf0a4ad777efc06dcf24cf09ac03718c4eb678c553855f4a4c', 'ml/forecast/test_route_specific.py': 'a4e356ccbfafade57b8bd33821f63895a866c13f98ba80832bead1caeba7b5f5', 'ml/forecast/run.py': 'ce421c6fc8d77406a3817c0a15253f3464d360cc7c69bb78e8bd131edef46072', 'ml/forecast/prepare_inputs.py': '9fdb609444a3e443961504e76bda2ccb1d02dadb3ad88bb0420f70fe01711dd5', 'ml/forecast/test_pipeline.py': '267a014e6b267f4c792b7a90c8679c98ee94e3e36e070baa706a89f5fa8f399b', 'ml/forecast/download_yandex.py': '300dec623de4bf983a7960e23bbd85a128c120d40e25059d1a000d898c2ff37d', 'ml/forecast/test_download_yandex.py': '9ca4e25061931b50b2b1730e4085f44add1c9d9a0071cd1290779e5b939666d1', 'ml/submit/adapter.py': '9767bfd9573ea7637669e37bac42e75e45205f29d533d24d1d397f646047d97d', 'ml/forecast/weather_competition.py': 'cd21d93a65ae8f8ee66f54005ece5063e26e15cff64cd64c4e6dcf45e1d4dc11', 'ml/forecast/reference_route_model.json': 'd346c229308017f192e614a6dde8a1f88026c014325e664fe8ed611325baf894', 'ml/forecast/test_weather_competition.py': '69a0829537f335cae8d4f00e218fff3af3030069fbba319a94343791693db09d', 'ml/forecast/weather_source.json': '0a6f625573c314cfde5991f1da6c505dd5c74841141e7c94c07a9594bcbee3aa', 'ml/forecast/network_regimes.py': '4cdf72b14761f337a68d191ee8aaae816bc14cdc089170bb20199e5003e32116', 'ml/forecast/network_competition.py': '402fdea43a4595160364919c89a97cf3f6a06ebb1a796030e34e58a161057ea8', 'ml/forecast/test_network_competition.py': '917561321305a30d7abcf0205212c74a7c3f32a70981c4eee9662dd4df638161', 'ml/forecast/network_sources.json': 'e3443edfa338b44d93430f39741fb6f7b924fca00475b8642cba43d10b44ca6a', 'ml/forecast/research_audit.json': 'f947e15e786cc979ac9acf57c11f153f3c7fdffc8828aaa3a620dbd34a4821cb'}
for name, source in SOURCES.items():
    if hashlib.sha256(source.encode()).hexdigest() != SOURCE_HASHES[name]:
        raise RuntimeError('Изменился встроенный исходник: ' + name)
    target = RUNTIME / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
print('Развёрнуты и проверены', len(SOURCES), 'модулей DS‑1.')

def run_module(module, *arguments):
    command = [sys.executable, '-u', '-m', module, *map(str, arguments)]
    with subprocess.Popen(command, cwd=RUNTIME, env=os.environ.copy(),
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        result = process.wait()
    if result:
        raise RuntimeError(f'{module} завершился с ошибкой {result}. См. вывод выше.')

if RUN_TESTS:
    run_module('unittest', 'discover', '-s', 'ml/forecast', '-p', 'test_*.py', '-v')


## 3. Скачать архив с Яндекс Диска и найти входы
Официальный публичный API не требует токена. Скачивание идёт потоком в `/tmp`,
с прогрессом, повторными попытками, проверкой размера и хеша. При повторном запуске
проверенный архив используется снова; частичная загрузка докачивается, если сервер
поддерживает Range. В память целый архив не загружается.

Распаковываются только небольшие метки и шаблон; сырые транзакции остаются в ZIP.
При сетевой ошибке ячейка остановится с объяснением, а не начнёт обучение без данных.
Официальный способ получения ссылки: https://yandex.cloud/en/docs/datasphere/operations/data/connect-to-ya-disk


In [ ]:
SEARCH_ROOT = INPUT_ROOT
if DATA_SOURCE == 'yandex':
    import importlib.util
    spec = importlib.util.spec_from_file_location('ds1_yandex', RUNTIME / 'ml/forecast/download_yandex.py')
    downloader = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(downloader)
    downloaded_archive = downloader.download(YANDEX_URL, DOWNLOAD_CACHE)
    SEARCH_ROOT = downloaded_archive.parent
elif DATA_SOURCE != 'input':
    raise ValueError("DATA_SOURCE должен быть 'yandex' или 'input'")
if not SEARCH_ROOT.is_dir():
    raise FileNotFoundError('Нет входной папки. Для ручного режима добавьте набор через Add Input.')
required = {'train': 'labels_day_train.csv', 'test': 'labels_day_test.csv',
            'sample': 'test_submission.csv'}
files = {}
for role, name in required.items():
    if INPUT_FILES[role]:
        path = Path(INPUT_FILES[role])
        if not path.is_file():
            raise FileNotFoundError(path)
        files[role] = path
    else:
        matches = sorted(SEARCH_ROOT.rglob(name))
        if len(matches) > 1:
            raise RuntimeError(f'Несколько {name}: {matches}. Укажите INPUT_FILES[{role!r}].')
        if matches:
            files[role] = matches[0]

missing = set(required) - set(files)
archive_candidates = {role: [] for role in missing}
if missing:
    for archive in sorted(SEARCH_ROOT.rglob('*.zip')):
        with zipfile.ZipFile(archive) as bundle:
            for info in bundle.infolist():
                for role in missing:
                    if Path(info.filename).name == required[role]:
                        archive_candidates[role].append((archive, info.filename, info.file_size))
    for role in sorted(missing):
        matches = archive_candidates[role]
        if len(matches) != 1:
            raise RuntimeError(f'{required[role]}: найдено вариантов в ZIP: {len(matches)}. '
                               'Добавьте этот файл отдельно и укажите путь в INPUT_FILES.')
        archive, member, size = matches[0]
        if size > 20_000_000:
            raise ValueError('Неожиданно большой файл меток/шаблона: ' + member)
        with zipfile.ZipFile(archive) as bundle:
            path = EXTRACTED / required[role]
            path.write_bytes(bundle.read(member))
            files[role] = path

for role, path in files.items():
    print(role, '→', path, '|', f'{path.stat().st_size:,}', 'байт')
# Описание внутри исходного ZIP тоже можно прочитать без извлечения raw CSV.
for archive in sorted(SEARCH_ROOT.rglob('*.zip')):
    with zipfile.ZipFile(archive) as bundle:
        descriptions_in_zip = [i for i in bundle.infolist()
                               if Path(i.filename).name.lower() == 'readme.md' and i.file_size < 200_000]
        if len(descriptions_in_zip) == 1:
            description_path = EXTRACTED / 'README.md'
            description_path.write_bytes(bundle.read(descriptions_in_zip[0]))
            print('Описание из архива:\n', description_path.read_text(encoding='utf-8-sig'))
descriptions = sorted(SEARCH_ROOT.rglob('README.md'))
if len(descriptions) == 1 and descriptions[0].stat().st_size < 200_000:
    print('\nОписание набора:\n', descriptions[0].read_text(encoding='utf-8-sig'))
else:
    print('Описание не показано автоматически; сверьте README организаторов отдельно.')


## Сравнение на настоящих метках

Три прогноза на 61/62 дня: май–июнь, июль–август, сентябрь–октябрь.
Дополнительно — **31 день октября** после обучения по 30 сентября:
новый сентябрьский режим маршрута 7 впервые имеет собственную историю.
Эти оценки подписаны раздельно. Погодная поправка должна улучшать каждый
месяц сентября–октября относительно контроля без погоды.

Проверяется также восстановление в августе: эксперимент DS2 показал,
что простое удаление укороченной истории ухудшает прогноз.


In [ ]:
import base64, zlib
WEATHER = EXTRACTED / 'moscow_2025_era5.json'
WEATHER.write_bytes(zlib.decompress(base64.b64decode('...')))
assert hashlib.sha256(WEATHER.read_bytes()).hexdigest() == 'e6330c286fde16bdd97ed815376b7d221a383e3ad8a041a51b3df70bef80f9e0'
run_module('ml.forecast.network_competition', '--train', files['train'], '--test', files['test'],
           '--sample', files['sample'], '--weather', WEATHER,
           '--reference', RUNTIME / 'ml/forecast/reference_route_model.json', '--out', OUT)


## Результаты и файлы

- **submission_network_weather.csv** — выбранный вариант по маршрутам.
- **submission_network_no_weather.csv** — тот же выбор без погодной поправки.
- **submission_regimes_candidate.csv** — режимы 7/50 с фиксированным смешиванием,
  без погодного бустинга; исследовательский вариант, даже если фильтр его отклонил.
- **submission_reference.csv** — прежний прогноз 0,877 без изменений.
- **submission_holiday_service_scenario.csv** — отдельный сценарий для 1/3/4 ноября:
  рабочая суббота без ремонта, праздничные будни с режимом воскресенья.
  Фактический график ремонта на эти даты не подтверждён; это не основной сабмит.

Все файлы сохраняют 14640 строк и маршрут 5 с нулями.
Восстановление с 15 ноября подтверждено новостью; эффект на число поездок
ещё не проверен по ноябрьским меткам. Ночные ограничения не зануляются
по неподтверждённому расписанию.


In [ ]:
from IPython.display import display, Markdown, FileLink
import shutil
report = json.loads((OUT/'network_weather_report.json').read_text())
lines = ['| Период | Вариант | WAPE-score |', '|---|---|---:|']
shown = ['plain/reference', 'only_50', 'only_7', 'network/reference',
         'plain/calendar_1.0', 'plain/weather_1.0',
         'network/calendar_1.0', 'network/weather_1.0', 'selected_no_weather', 'selected']
for period, fold in report['folds'].items():
    for name in shown:
        if name in fold['variants']:
            m = fold['variants'][name]['all']['wape_score']
            lines.append(f"| {period} | {name} | {m:.6f} |")
display(Markdown('\n'.join(lines)))
print('Режимы приняты на маршрутах:', report['selection']['network_accepted_routes'])
print('Погода принята на маршрутах:', report['selection']['weather_accepted_routes'])
print('Выбор:', report['selection']['by_route'])
print('Октябрь, дополнительные параметры только по сентябрю:',
      report['october_forward_from_september']['metrics']['all']['wape_score'])
print('Изменено строк:', report['changed_rows_vs_reference'])
print('Платформенный score новой модели пока неизвестен.')
for name in ['network_weather_report.json', *report['outputs']]:
    target = WORK/name
    shutil.copy2(OUT/name, target)
    display(FileLink(str(target.relative_to(Path.cwd())) if target.is_relative_to(Path.cwd()) else str(target)))
# Малые настоящие входы для следующего локального исследования; не публиковать.
for role, path in files.items():
    target = WORK/('organizer_'+path.name)
    shutil.copy2(path, target)
    display(FileLink(str(target.relative_to(Path.cwd())) if target.is_relative_to(Path.cwd()) else str(target)))
